<a href="https://colab.research.google.com/github/Glory-K/test/blob/main/%EB%8C%80%EA%B5%AC%EA%B4%91%EC%97%AD%EC%8B%9C_%EC%9E%90%EC%B9%98%EA%B5%AC_%EB%8B%A8%EC%9C%84_CCTV%2C_%EC%9D%B8%EA%B5%AC%2C_%EC%9A%A9%EB%8F%84%EC%A7%80%EC%97%AD%EA%B3%BC_%EB%B2%94%EC%A3%84_%EB%B0%9C%EC%83%9D_%ED%8A%B9%EC%84%B1_%EB%B6%84%EC%84%9D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 0. 라이브러리 설치 및 호출

### 0-1. 폰트 설치

In [ ]:
!apt-get update -qq
!apt-get install -y fonts-nanum -qq
!rm -rf ~/.cache/matplotlib

### 0-2. 라이브러리 설치

In [ ]:
!pip install -q geopandas pyogrio shapely fiona

### 0-3. 라이브러리 호출

In [ ]:
import os
import re
import time
import requests
import warnings
from typing import Dict, List


import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from tqdm.auto import tqdm

from google.colab import drive

warnings.filterwarnings("ignore")

### 0-4. 기본 설정

##### 0-4-1. 시각화 설정

In [ ]:
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 200)

!fc-cache -fv > /dev/null 2>&1

font_paths = fm.findSystemFonts(fontpaths=None, fontext="ttf")
nanum_fonts = [f for f in font_paths if "Nanum" in f or "nanum" in f]

if nanum_fonts:
    font_name = fm.FontProperties(fname=nanum_fonts[0]).get_name()
    plt.rc("font", family=font_name)
    print(f"적용된 한글 폰트: {font_name}")
else:
    print("나눔 폰트를 찾지 못했습니다. 런타임 재시작 후 다시 실행해보세요.")

plt.rcParams["axes.unicode_minus"] = False

In [ ]:
plt.figure(figsize=(6, 4))
plt.title("대구광역시 자치구별 예시 그래프")
plt.bar(["중구", "동구", "서구"], [10, 20, 15])
plt.xlabel("자치구")
plt.ylabel("값")
plt.show()

##### 0-4-2. 기본 설정

In [ ]:
# 연도 목록
YEARS = list(range(2019, 2025))

# 대구 자치구 코드/이름
DAEGU_SIG_MAP = {
    "27110": "중구",
    "27140": "동구",
    "27170": "서구",
    "27200": "남구",
    "27230": "북구",
    "27260": "수성구",
    "27290": "달서구",
    "27710": "달성군",
    "27720": "군위군",
}

# 장기 비교(2019~2024)에서는 군위군 제외
DAEGU_SIG_MAP_LONG = {
    "27110": "중구",
    "27140": "동구",
    "27170": "서구",
    "27200": "남구",
    "27230": "북구",
    "27260": "수성구",
    "27290": "달서구",
    "27710": "달성군",
}

DAEGU_GU_LIST = list(DAEGU_SIG_MAP.values())
DAEGU_GU_LIST_LONG = list(DAEGU_SIG_MAP_LONG.values())

# 1. 데이터 로드

### 1-1. Google Drive 연결 및 경로 설정

In [ ]:
drive.mount("/content/drive")

base_path = "/content/drive/MyDrive/Colab_Notebooks/data/대구"

population_path = os.path.join(base_path, "population")
crime_path = os.path.join(base_path, "crime")
cctv_path = os.path.join(base_path, "cctv")
landuse_path = os.path.join(base_path, "land")
geo_path = os.path.join(base_path, "geo")

output_path = "/content/drive/MyDrive/Colab_Notebooks/data/outputs"
os.makedirs(output_path, exist_ok=True)

print("기본 경로 설정 완료")
print("base_path:", base_path)
print("output_path:", output_path)

### 1-2. 공통 유틸 함수

In [ ]:
def safe_read_csv(file_path: str, encodings: List[str] = None) -> pd.DataFrame:
    """
    인코딩 이슈가 있을 수 있는 CSV를 안전하게 읽기 위한 함수
    """
    if encodings is None:
        encodings = [None, "utf-8", "utf-8-sig", "cp949", "euc-kr"]

    errors = []

    for enc in encodings:
        try:
            if enc is None:
                return pd.read_csv(file_path)
            else:
                return pd.read_csv(file_path, encoding=enc)
        except UnicodeDecodeError as e:
            errors.append((enc, type(e).__name__, str(e)[:120]))
        except pd.errors.ParserError as e:
            errors.append((enc, type(e).__name__, str(e)[:120]))
        except Exception as e:
            errors.append((enc, type(e).__name__, str(e)[:120]))

    raise ValueError(f"파일을 읽을 수 없습니다: {file_path}\n시도 결과: {errors}")


def print_shape_summary(name: str, data_dict: Dict):
    print(f"\n[{name}]")
    for k, v in data_dict.items():
        print(f"{k}: {v.shape}")


def extract_yearly_population_columns(df: pd.DataFrame, year: int) -> List[str]:
    """
    해당 연도의 월별 총인구수 컬럼만 추출
    예: 2024년01월_총인구수 ~ 2024년12월_총인구수
    """
    pattern = rf"^{year}년\d{{2}}월_총인구수$"
    return [col for col in df.columns if re.match(pattern, col)]


def extract_admin_name(text: str) -> str:
    """
    예: '대구광역시 중구 (2711000000)' -> '중구'
    예: '대구광역시  (2700000000)' -> '대구광역시'
    """
    if pd.isna(text):
        return np.nan

    text = str(text).strip()

    if text.startswith("대구광역시 ") and "(" in text:
        core = text.split("(")[0].strip()
        core = core.replace("대구광역시", "").strip()
        return core if core else "대구광역시"

    return text

### 1-3. 인구 데이터 로드

In [ ]:
population_dfs = {}
for year in YEARS:
    file_name = f"{year}01_{year}12_주민등록인구및세대현황_월간.csv"
    file_path = os.path.join(population_path, file_name)
    population_dfs[year] = safe_read_csv(file_path, encodings=["cp949", "utf-8-sig", "utf-8"])

print_shape_summary("인구 데이터", population_dfs)

### 1-4. 범죄 데이터 로드

In [ ]:
crime_dfs = {}
for year in YEARS:
    file_name = f"범죄발생지_{year}.csv"
    file_path = os.path.join(crime_path, file_name)
    crime_dfs[year] = safe_read_csv(file_path, encodings=["cp949", "utf-8-sig", "utf-8"])

print_shape_summary("범죄 데이터", crime_dfs)

### 1-5. CCTV 데이터 로드

In [ ]:
cctv_origin_path = os.path.join(base_path, "cctv_origin")

cctv_origin_files = {
    "중구":   "대구_중구.xlsx",
    "동구":   "대구_동구.xlsx",
    "서구":   "CCTV정보_대구서구.csv",
    "남구":   "대구_남구.xlsx",
    "북구":   "대구_북구.xlsx",
    "수성구": "CCTV정보_대구수성구.csv",
    "달서구": "대구_달서구.csv",
    "달성군": "대구_달성군.xlsx",
    "군위군": "대구_군위군.xlsx",
}

def read_csv_safe(file_path: str) -> pd.DataFrame:
    encodings = [None, "utf-8", "utf-8-sig", "cp949", "euc-kr"]
    errors = []

    for enc in encodings:
        try:
            if enc is None:
                return pd.read_csv(file_path)
            return pd.read_csv(file_path, encoding=enc)
        except Exception as e:
            errors.append((enc, type(e).__name__, str(e)[:120]))

    raise ValueError(f"CSV 파일을 읽을 수 없습니다: {file_path}\n시도 결과: {errors}")

def read_excel_safe(file_path: str) -> pd.DataFrame:
    try:
        return pd.read_excel(file_path)
    except Exception as e:
        raise ValueError(f"Excel 파일을 읽을 수 없습니다: {file_path}\n에러: {e}")

def load_cctv_origin(file_path: str) -> pd.DataFrame:
    ext = os.path.splitext(file_path)[1].lower()

    if ext == ".csv":
        return read_csv_safe(file_path)
    elif ext in [".xlsx", ".xls"]:
        return read_excel_safe(file_path)
    else:
        raise ValueError(f"지원하지 않는 파일 형식입니다: {file_path}")

# 원본 CCTV 로드
cctv_origin_dfs = {}
for gu, file_name in cctv_origin_files.items():
    file_path = os.path.join(cctv_origin_path, file_name)
    cctv_origin_dfs[gu] = load_cctv_origin(file_path)

# 이후 기존 코드와 호환되도록 cctv_dfs에 덮어쓰기
cctv_dfs = cctv_origin_dfs.copy()

print("[CCTV 원본 데이터 shape]")
for gu, df in cctv_dfs.items():
    print(f"{gu}: {df.shape}")

### 1-6. 용도지역 데이터 로드

In [ ]:
landuse_dfs = {}
for year in YEARS:
    file_name = f"대구 용도지역 현황_{year}.csv"
    file_path = os.path.join(landuse_path, file_name)
    landuse_dfs[year] = safe_read_csv(file_path, encodings=["cp949", "utf-8-sig", "utf-8"])

print_shape_summary("용도지역 데이터", landuse_dfs)

### 1-7. 법정구역 경계 데이터 로드

In [ ]:
geo_file_map = {
    2019: "AL_00_D001_20191207/AL_00_D001_20191207(SIG)/AL_00_D001_20191207(SIG).shp",
    2020: "AL_00_D001_20201205/AL_00_D001_20201205(SIG)/AL_00_D001_20201205(SIG).shp",
    2021: "AL_00_D001_20211204/AL_00_D001_20211204(SIG)/AL_00_D001_20211204(SIG).shp",
    2022: "AL_00_D001_20221203/AL_00_D001_20221203(SIG)/AL_00_D001_20221203(SIG).shp",
    2023: "AL_D001_00_20231204/AL_D001_00_20231204(SIG)/AL_D001_00_20231204(SIG).shp",
    2024: "AL_D001_00_20241204/AL_D001_00_20241204(SIG)/AL_D001_00_20241204(SIG).shp",
}

geo_dfs = {}
for year, rel_path in geo_file_map.items():
    file_path = os.path.join(geo_path, rel_path)
    geo_dfs[year] = gpd.read_file(file_path, encoding="cp949")

print_shape_summary("법정구역 경계 데이터", geo_dfs)

### 1-8. 편의용 별칭(핵심 분석 중심 연도: 2024)

In [ ]:
pop = population_dfs[2024]
crime = crime_dfs[2024]
landuse = landuse_dfs[2024]
shp2024 = geo_dfs[2024]

print("2024년 대표 데이터 별칭 생성 완료")

# 2. 데이터 탐색

### 2-1. 전체 구조 확인

##### 2-1-1. 데이터 탐색

In [ ]:
print("\n[인구 데이터 2024]")
display(pop.head())

print("\n[범죄 데이터 2024]")
display(crime.head(10))

print("\n[CCTV 데이터 - 중구 예시]")
display(cctv_dfs["중구"].head())

print("\n[용도지역 데이터 2024]")
display(landuse.head())

print("\n[법정경계 데이터 2024]")
display(shp2024.head())

##### 2-1-2. 칼럼명 확인

In [ ]:
print("\n[인구 컬럼]")
print(pop.columns.tolist())

print("\n[범죄 컬럼]")
print(crime.columns.tolist())

print("\n[CCTV 중구 컬럼]")
print(cctv_dfs["중구"].columns.tolist())

print("\n[용도지역 컬럼]")
print(landuse.columns.tolist())

print("\n[법정경계 컬럼]")
print(shp2024.columns.tolist())

### 2-2. 인구 데이터 탐색

##### 2-2-1. 인구 데이터 탐색

In [ ]:
for year in YEARS:
    temp = population_dfs[year].copy()
    temp["행정구역_정리"] = temp["행정구역"].apply(extract_admin_name)
    pop_cols = extract_yearly_population_columns(temp, year)

    print(f"\n===== 인구 데이터 점검: {year} =====")
    print("shape:", temp.shape)
    print("행정구역 예시:", temp["행정구역"].head(10).tolist())
    print("정리된 행정구역 예시:", temp["행정구역_정리"].head(10).tolist())
    print("월별 총인구수 컬럼 수:", len(pop_cols))
    print("월별 총인구수 컬럼:", pop_cols[:3], "...", pop_cols[-3:] if len(pop_cols) >= 3 else pop_cols)

##### 2-2-2. 2024년 인구 데이터에서 대구 전체/자치구 확인

In [ ]:
pop_2024_check = pop.copy()
pop_2024_check["행정구역_정리"] = pop_2024_check["행정구역"].apply(extract_admin_name)

print("\n[2024 인구 데이터 - 정리된 행정구역 목록]")
display(pop_2024_check[["행정구역", "행정구역_정리"]].head(15))

##### 2-2-3. 2024년 월별 총인구수 칼럼 확인 및 숫자형 변환 가능성 탐색

In [ ]:
pop_2024_total_cols = extract_yearly_population_columns(pop, 2024)
print("2024 총인구수 컬럼:", pop_2024_total_cols)

pop_numeric_preview = pop[["행정구역"] + pop_2024_total_cols[:3]].copy()
display(pop_numeric_preview.head())

for col in pop_2024_total_cols[:3]:
    print(f"{col} dtype:", pop[col].dtype)

### 2-3. 범죄 데이터 탐색

##### 2-3-1. 범죄 데이터 탐색

In [ ]:
for year in YEARS:
    temp = crime_dfs[year]
    print(f"\n===== 범죄 데이터 점검: {year} =====")
    print("shape:", temp.shape)
    display(temp.head(5))

##### 2-3-2. 2024 범죄 데이터의 상단 구조 확인

In [ ]:
print("\n[범죄 데이터 상단 구조 확인]")
display(crime.iloc[:10, :12])

##### 2-3-3. 2024 범죄 데이터에서 첫 두 행 별도 확인

In [ ]:
print("\n[범죄 데이터 1행]")
display(crime.iloc[[0]])

print("\n[범죄 데이터 2행]")
display(crime.iloc[[1]])

##### 2-3-4. 2024 범죄 데이터에서 자치구 헤더 후보 확인

In [ ]:
print("\n[범죄 데이터 2024 - 컬럼별 상위 3행 확인]")
crime_header_check = pd.DataFrame({
    "col_name": crime.columns,
    "row0": crime.iloc[0].values,
    "row1": crime.iloc[1].values,
})
display(crime_header_check)

##### 2-3-5. 범죄 소분류(범죄별(3)) 후보 확인

In [ ]:
third_col = crime.columns[2]
print("3번째 컬럼명:", third_col)
print("\n[범죄 소분류 예시]")
display(crime[[crime.columns[0], crime.columns[1], crime.columns[2]]].head(30))

### 2-4. CCTV 데이터 탐색

##### 2-4-0. 보조 함수 및 칼럼 후보 정의

In [ ]:
def clean_numeric_series(series: pd.Series) -> pd.Series:
    """
    문자열/숫자 혼합 컬럼을 안전하게 숫자형으로 변환
    """
    s = series.copy()
    s = s.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })
    s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")

def extract_install_year(series: pd.Series) -> pd.Series:
    """
    설치연도/설치일/설치년월 등에서 연도만 추출
    """
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })

    # 1차: 숫자형 그대로 시도
    num = pd.to_numeric(s, errors="coerce")
    year_num = num.where(num.between(1900, 2100), pd.NA)

    # 2차: 문자열 내 4자리 연도 추출
    year_str = s.str.extract(r"(19\d{2}|20\d{2})", expand=False)
    year_str = pd.to_numeric(year_str, errors="coerce")

    return year_num.fillna(year_str)

def normalize_text_series(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })
    return s

def find_existing_cols(df: pd.DataFrame, candidates: list[str]) -> list[str]:
    return [c for c in candidates if c in df.columns]

def coalesce_series(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    """
    여러 후보 컬럼 중 존재하는 컬럼을 왼쪽부터 우선 사용
    """
    found = [c for c in candidates if c in df.columns]

    if not found:
        return pd.Series([pd.NA] * len(df), index=df.index, dtype="object")

    out = df[found[0]].copy()
    for c in found[1:]:
        out = out.where(out.notna() & (out.astype("string").str.strip() != ""), df[c])

    return out

CCTV_FIELD_CANDIDATES = {
    "소재지": [
        "카메라 설치위치",
        "설치장소(도로명주소)",
        "소재지도로명주소",
        "소재지지번주소",
        "카메라 설치위치 ",
        "주소",
    ],
    "설치목적구분": [
        "설치목적구분", "설치목적", "설치 목적", "설치목적", "목적", "용도"
    ],
    "카메라수": [
        "카메라수", "카메라대수", "카메라 대수"
    ],
    "설치연도": [
        "설치연도", "설치년도", "설치연월", "설치년월", "설치일자", "설치일", "설치년", "설치연월"
    ],
    "위도": [
        "위도", "WGS84위도", "Latitude", "latitude", "lat"
    ],
    "경도": [
        "경도", "WGS84경도", "Longitude", "longitude", "lon", "lng"
    ],
}

##### 2-4-1. 자치구별 CCTV 데이터 기본 구조 확인

In [ ]:
for gu, df in cctv_dfs.items():
    print(f"\n===== CCTV 데이터 점검: {gu} =====")
    print("shape:", df.shape)
    display(df.head(3))

##### 2-4-2. CCTV 칼럼 비교

In [ ]:
cctv_col_summary = pd.DataFrame({
    gu: pd.Series(df.columns.tolist()) for gu, df in cctv_dfs.items()
})

print("[CCTV 원본 자치구별 컬럼 비교]")
display(cctv_col_summary)

##### 2-4-3. CCTV 핵심 칼럼 존재 여부 확인

In [ ]:
mapping_rows = []

for gu, df in cctv_dfs.items():
    row = {"자치구": gu, "shape": df.shape}
    for std_col, candidates in CCTV_FIELD_CANDIDATES.items():
        found = find_existing_cols(df, candidates)
        row[std_col] = ", ".join(found) if found else None
    mapping_rows.append(row)

cctv_mapping_summary_df = pd.DataFrame(mapping_rows)

print("[CCTV 의미 기준 컬럼 매핑 요약]")
display(cctv_mapping_summary_df)

##### 2-4-4. CCTV 설치목적구분 고유값 확인

In [ ]:
purpose_rows = []

for gu, df in cctv_dfs.items():
    purpose_raw = coalesce_series(df, CCTV_FIELD_CANDIDATES["설치목적구분"])
    purpose_raw = normalize_text_series(purpose_raw)

    unique_values = sorted(
        purpose_raw.dropna().astype(str).unique().tolist()
    )

    purpose_rows.append({
        "자치구": gu,
        "설치목적개수": len(unique_values),
        "설치목적목록": unique_values
    })

cctv_origin_purpose_unique_df = pd.DataFrame(purpose_rows)

print("[CCTV 원본 설치목적 고유값]")
display(cctv_origin_purpose_unique_df)

##### 2-4-5. CCTV 설치연도 품질 점검

In [ ]:
year_quality_rows = []

for gu, df in cctv_dfs.items():
    year_raw = coalesce_series(df, CCTV_FIELD_CANDIDATES["설치연도"])
    year_num = extract_install_year(year_raw)

    total_n = len(df)
    exist_n = year_num.notna().sum()
    miss_n = year_num.isna().sum()

    year_quality_rows.append({
        "자치구": gu,
        "전체행수": total_n,
        "설치연도존재행수": int(exist_n),
        "설치연도결측행수": int(miss_n),
        "설치연도존재비율": round(exist_n / total_n, 4) if total_n > 0 else np.nan
    })

cctv_origin_year_quality_df = (
    pd.DataFrame(year_quality_rows)
    .sort_values("자치구")
    .reset_index(drop=True)
)

print("[CCTV 원본 설치연도 품질 점검]")
display(cctv_origin_year_quality_df)

##### 2-4-6. CCTV 카메라수 품질 점검

In [ ]:
camera_quality_rows = []

for gu, df in cctv_dfs.items():
    cam_raw = coalesce_series(df, CCTV_FIELD_CANDIDATES["카메라수"])
    cam_num = clean_numeric_series(cam_raw)

    total_n = len(df)
    exist_n = cam_num.notna().sum()
    miss_n = cam_num.isna().sum()

    camera_quality_rows.append({
        "자치구": gu,
        "전체행수": total_n,
        "카메라수존재행수": int(exist_n),
        "카메라수결측행수": int(miss_n),
        "카메라수존재비율": round(exist_n / total_n, 4) if total_n > 0 else np.nan
    })

cctv_origin_camera_quality_df = (
    pd.DataFrame(camera_quality_rows)
    .sort_values("자치구")
    .reset_index(drop=True)
)

print("[CCTV 원본 카메라수 품질 점검]")
display(cctv_origin_camera_quality_df)

##### 2-4-7. CCTV 좌표 결측 점검

In [ ]:
coord_quality_rows = []

for gu, df in cctv_dfs.items():
    lat_raw = coalesce_series(df, CCTV_FIELD_CANDIDATES["위도"])
    lon_raw = coalesce_series(df, CCTV_FIELD_CANDIDATES["경도"])

    lat_num = clean_numeric_series(lat_raw)
    lon_num = clean_numeric_series(lon_raw)

    total_n = len(df)
    coord_exist_n = (lat_num.notna() & lon_num.notna()).sum()
    coord_miss_n = total_n - coord_exist_n

    coord_quality_rows.append({
        "자치구": gu,
        "전체행수": total_n,
        "좌표존재행수": int(coord_exist_n),
        "좌표결측행수": int(coord_miss_n),
        "좌표존재비율": round(coord_exist_n / total_n, 4) if total_n > 0 else np.nan
    })

cctv_origin_coord_quality_df = (
    pd.DataFrame(coord_quality_rows)
    .sort_values("자치구")
    .reset_index(drop=True)
)

print("[CCTV 원본 좌표 품질 점검]")
display(cctv_origin_coord_quality_df)

##### 2-4-8. CCTV 공통 의미 칼럼 기준으로 1차 통합

In [ ]:
cctv_origin_unified_list = []

for gu, df in cctv_dfs.items():
    temp = df.copy()
    temp.columns = [str(c).strip() for c in temp.columns]

    unified = pd.DataFrame({
        "자치구": gu,
        "소재지_raw": normalize_text_series(coalesce_series(temp, CCTV_FIELD_CANDIDATES["소재지"])),
        "설치목적구분_raw": normalize_text_series(coalesce_series(temp, CCTV_FIELD_CANDIDATES["설치목적구분"])),
        "카메라수_raw": coalesce_series(temp, CCTV_FIELD_CANDIDATES["카메라수"]),
        "설치연도_raw": coalesce_series(temp, CCTV_FIELD_CANDIDATES["설치연도"]),
        "위도_raw": coalesce_series(temp, CCTV_FIELD_CANDIDATES["위도"]),
        "경도_raw": coalesce_series(temp, CCTV_FIELD_CANDIDATES["경도"]),
    })

    cctv_origin_unified_list.append(unified)

cctv_origin_unified_df = pd.concat(cctv_origin_unified_list, ignore_index=True)

print("[cctv_origin_unified_df]")
display(cctv_origin_unified_df.head(20))

print("\n[cctv_origin_unified_df shape]")
print(cctv_origin_unified_df.shape)

##### 2-4-9. CCTV 좌표 기초 통계

In [ ]:
cctv_origin_coord_stats_df = cctv_origin_unified_df.copy()
cctv_origin_coord_stats_df["위도_num"] = clean_numeric_series(cctv_origin_coord_stats_df["위도_raw"])
cctv_origin_coord_stats_df["경도_num"] = clean_numeric_series(cctv_origin_coord_stats_df["경도_raw"])

valid_coord_df = cctv_origin_coord_stats_df[
    cctv_origin_coord_stats_df["위도_num"].notna() &
    cctv_origin_coord_stats_df["경도_num"].notna()
].copy()

print("[원본 CCTV 좌표 유효행 수]")
print(len(valid_coord_df))

if len(valid_coord_df) > 0:
    print("\n[위도 describe]")
    display(valid_coord_df["위도_num"].describe())

    print("\n[경도 describe]")
    display(valid_coord_df["경도_num"].describe())
else:
    print("원본 좌표가 존재하는 행이 없습니다.")

##### 2-4-10. CCTV 좌표 이상치 후보 확인(단순 IQR 기준)

In [ ]:
if len(valid_coord_df) > 0:
    valid_coord_df["좌표범위이상여부"] = (
        (valid_coord_df["위도_num"] < 33) | (valid_coord_df["위도_num"] > 39) |
        (valid_coord_df["경도_num"] < 124) | (valid_coord_df["경도_num"] > 132)
    )

    print("[범위 기준 이상치 개수]")
    print(valid_coord_df["좌표범위이상여부"].sum())

    display(
        valid_coord_df.loc[
            valid_coord_df["좌표범위이상여부"],
            ["자치구", "소재지_raw", "설치목적구분_raw", "위도_raw", "경도_raw", "위도_num", "경도_num"]
        ].head(20)
    )
else:
    print("원본 좌표가 없어 이상치 점검을 수행하지 않습니다.")

##### 2-4-11. 최소/최대 좌표 행 확인

In [ ]:
if len(valid_coord_df) > 0:
    print("[최대 위도 행]")
    display(valid_coord_df.loc[[valid_coord_df["위도_num"].idxmax()]])

    print("\n[최소 위도 행]")
    display(valid_coord_df.loc[[valid_coord_df["위도_num"].idxmin()]])

    print("\n[최대 경도 행]")
    display(valid_coord_df.loc[[valid_coord_df["경도_num"].idxmax()]])

    print("\n[최소 경도 행]")
    display(valid_coord_df.loc[[valid_coord_df["경도_num"].idxmin()]])
else:
    print("원본 좌표가 없어 극값 확인을 수행하지 않습니다.")

##### 2-4-12. 설치연도/카메라수 분포 확

In [ ]:
cctv_origin_dist_df = cctv_origin_unified_df.copy()
cctv_origin_dist_df["카메라수_num"] = clean_numeric_series(cctv_origin_dist_df["카메라수_raw"])
cctv_origin_dist_df["설치연도_num"] = extract_install_year(cctv_origin_dist_df["설치연도_raw"])

print("[카메라수 describe]")
display(cctv_origin_dist_df["카메라수_num"].describe())

print("\n[설치연도 describe]")
display(cctv_origin_dist_df["설치연도_num"].describe())

### 2-5. 용도지역 데이터 탐색

##### 2-5-1. 용도지역 데이터 탐색

In [ ]:
for year in YEARS:
    temp = landuse_dfs[year]
    print(f"\n===== 용도지역 데이터 점검: {year} =====")
    print("shape:", temp.shape)
    display(temp.head(5))

##### 2-5-2. 용도지역의 구분/상세구분 확인

In [ ]:
print("\n[용도지역 구분 고유값]")
display(pd.Series(landuse["구분"].dropna().unique()).sort_values().reset_index(drop=True))

print("\n[용도지역 상세구분 예시]")
display(pd.Series(landuse["상세구분"].dropna().unique()).sort_values().reset_index(drop=True).head(50))

##### 2-5-3. 자치구 열 확인

In [ ]:
landuse_area_cols = [col for col in landuse.columns if col.endswith("(m2)")]
print("면적 컬럼:", landuse_area_cols)

##### 2-5-4. 군위군 존재 여부 점검 (2023년 편입 이슈 확인용)

In [ ]:
for year in YEARS:
    temp = landuse_dfs[year]
    cols = temp.columns.tolist()
    has_gunwi = "군위군(m2)" in cols
    print(f"{year}: 군위군 컬럼 존재 여부 -> {has_gunwi}")

### 2-6. 법정경계 데이터 탐색

##### 2-6-1. 법정경계 데이터 탐색

In [ ]:
for year, gdf in geo_dfs.items():
    print(f"\n===== 법정경계 데이터 점검: {year} =====")
    print("shape:", gdf.shape)
    print("crs:", gdf.crs)
    print("columns:", gdf.columns.tolist())
    display(gdf.head(3))

##### 2-6-2. 2024 경계 데이터에서 대구 SIG 필터링 준비

In [ ]:
print("\n[A1 상위 10개]")
display(shp2024["A1"].astype(str).head(10))

print("\n[A4 상위 10개]")
display(shp2024["A4"].astype(str).head(10))

##### 2-6-3. 대구 SIG 필터링 함수

In [ ]:
def filter_daegu_sig(gdf: gpd.GeoDataFrame, code_col: str = "A1") -> gpd.GeoDataFrame:
    code_list = list(DAEGU_SIG_MAP.keys())
    temp = gdf.copy()
    temp[code_col] = temp[code_col].astype(str)
    return temp[temp[code_col].isin(code_list)].copy()

shp2024_daegu = filter_daegu_sig(shp2024, code_col="A1")
print("2024 대구 경계 shape:", shp2024_daegu.shape)
display(shp2024_daegu[["A1", "A2", "A3", "A4", "geometry"]].head(20))

##### 2-6-4. 연도별 대구 경계 필터링 개수 확인

In [ ]:
geo_daegu_dfs = {}
for year, gdf in geo_dfs.items():
    temp = filter_daegu_sig(gdf, code_col="A1")
    geo_daegu_dfs[year] = temp
    print(f"{year}: 대구 SIG 개수 -> {len(temp)}")

##### 2-6-5. 군위군 편입 확인(장기 비교 전체 점검)

In [ ]:
for year, gdf in geo_daegu_dfs.items():
    names = gdf["A2"].astype(str).tolist()
    has_gunwi = "군위군" in names
    print(f"{year}: 군위군 포함 여부 -> {has_gunwi}")

##### 2-6-6. 2024년 대구 경계 시각 확인

In [ ]:
ax = shp2024_daegu.plot(figsize=(8, 8), edgecolor="black")
for idx, row in shp2024_daegu.iterrows():
    if row.geometry is not None and not row.geometry.is_empty:
        x, y = row.geometry.centroid.x, row.geometry.centroid.y
        ax.text(x, y, row["A2"], fontsize=9)

ax.set_title("2024년 대구 자치구 경계(탐색용)")
plt.show()

##### 2-6-7. 2023년과 2024년 남구 geometry 비교용 추출

In [ ]:
shp2023_daegu = geo_daegu_dfs[2023].copy()

namgu_2023 = shp2023_daegu[shp2023_daegu["A2"] == "남구"].copy()
namgu_2024 = shp2024_daegu[shp2024_daegu["A2"] == "남구"].copy()

print("2023 남구 개수:", len(namgu_2023))
print("2024 남구 개수:", len(namgu_2024))

display(namgu_2023[["A1", "A2", "geometry"]])
display(namgu_2024[["A1", "A2", "geometry"]])

##### 2-6-8. 2023/2024 남구 bounds 비교

In [ ]:
if len(namgu_2023) > 0 and len(namgu_2024) > 0:
    print("2023 남구 bounds:", namgu_2023.total_bounds)
    print("2024 남구 bounds:", namgu_2024.total_bounds)

### 2-7. 2024 핵심 분석용 데이터 구조 점검 요약

In [ ]:
summary_rows = []

summary_rows.append({
    "데이터셋": "인구(2024)",
    "shape": pop.shape,
    "핵심메모": "월별 wide 구조, 대구 전체 행 포함, 문자열 숫자 정리 후 연평균 인구 계산 필요"
})

summary_rows.append({
    "데이터셋": "범죄(2024)",
    "shape": crime.shape,
    "핵심메모": "다중 헤더 구조, 자치구명 재설정 필요, '-' 값 0 처리 및 long 변환 필요"
})

summary_rows.append({
    "데이터셋": "CCTV 원본 1차 통합(탐색용)",
    "shape": cctv_origin_unified_df.shape,
    "핵심메모": "자치구별 원본 파일 구조 상이, 공통 의미 컬럼 기준 1차 통합 완료, 카메라수/설치연도/좌표 품질 차이 큼"
})

summary_rows.append({
    "데이터셋": "용도지역(2024)",
    "shape": landuse.shape,
    "핵심메모": "행=세부용도, 열=자치구 wide 구조, long 변환 및 상위 구분(주거/상업/공업) 재집계 필요"
})

summary_rows.append({
    "데이터셋": "경계(2024-원본)",
    "shape": shp2024.shape,
    "핵심메모": "전국 SIG 전체 포함, 대구 자치구만 필터링 필요, CRS 및 컬럼 구조 점검 필요"
})

if "shp2024_daegu" in globals():
    summary_rows.append({
        "데이터셋": "경계(2024-대구 필터)",
        "shape": shp2024_daegu.shape,
        "핵심메모": "대구 9개 자치구 필터링 완료, 남구 geometry 오류 검토 및 면적 계산 전 CRS 통일 필요"
    })

eda_summary_df = pd.DataFrame(summary_rows)

print("[2024 핵심 분석용 데이터 구조 점검 요약]")
display(eda_summary_df)

# 3. 전처리

### 3-0. 전처리용 함수

##### 3-0-1. 공통함수

In [ ]:
def clean_numeric_series(series: pd.Series) -> pd.Series:
    s = series.copy()
    s = s.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })
    s = s.str.replace(",", "", regex=False)
    return pd.to_numeric(s, errors="coerce")


def normalize_text_series(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })
    return s


def extract_admin_name(text: str) -> str:
    if pd.isna(text):
        return np.nan

    text = str(text).strip()

    m = re.match(r"대구광역시\s*(.*?)\s*\(", text)
    if m:
        name = m.group(1).strip()
        return name if name else "대구광역시"

    return text


def standardize_gu_name(name: str) -> str:
    if pd.isna(name):
        return np.nan

    name = str(name).strip()

    name = re.sub(
        r"^(대구광역시|서울특별시|부산광역시|인천광역시|광주광역시|대전광역시|울산광역시|세종특별자치시|"
        r"경기도|강원특별자치도|충청북도|충청남도|전북특별자치도|전라남도|경상북도|경상남도|제주특별자치도)\s+",
        "",
        name
    )

    mapping = {
        "중구": "중구",
        "동구": "동구",
        "서구": "서구",
        "남구": "남구",
        "북구": "북구",
        "수성": "수성구",
        "수성구": "수성구",
        "달서": "달서구",
        "달서구": "달서구",
        "달성": "달성군",
        "달성군": "달성군",
        "군위": "군위군",
        "군위군": "군위군",
        "대구광역시": "대구광역시",
    }
    return mapping.get(name, name)


def extract_install_year(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.strip()
    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "NULL": pd.NA,
        "null": pd.NA,
        "-": pd.NA
    })

    num = pd.to_numeric(s, errors="coerce")
    year_num = num.where(num.between(1900, 2100), pd.NA)

    year_str = s.str.extract(r"(19\d{2}|20\d{2})", expand=False)
    year_str = pd.to_numeric(year_str, errors="coerce")

    return year_num.fillna(year_str)

##### 3-0-2. 5대범죄 재분류

In [ ]:
VIOLENCE_ITEMS = {
    "폭행",
    "상해",
    "협박",
    "공갈",
    "약취와 유인",
    "체포와 감금",
    "폭력행위등(손괴·강요·주거침입등)",
    "폭력행위등(단체등의구성·활동)",
}

def map_five_major_crime(crime_l3: str) -> str:
    if pd.isna(crime_l3):
        return np.nan

    crime_l3 = str(crime_l3).strip()

    if crime_l3 == "살인":
        return "살인"
    elif crime_l3 == "강도":
        return "강도"
    elif crime_l3 == "절도":
        return "절도"
    elif crime_l3 == "성폭력":
        return "성범죄"
    elif crime_l3 in VIOLENCE_ITEMS:
        return "폭력"
    else:
        return np.nan

##### 3-0-3. CCTV 설치목적 표준화

In [ ]:
def normalize_purpose_raw(x: str) -> str:
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\n", " ").strip()
    x = re.sub(r"\s+", " ", x)
    return x


def map_cctv_purpose(x: str) -> str:
    x = normalize_purpose_raw(x)

    if pd.isna(x):
        return np.nan

    if x in {"방범", "생활방범", "생활안전", "차량방범"}:
        return "방범"

    if x in {"어린이보호", "어린이보호구역", "영유아 안전및보안"}:
        return "어린이 보호구역"

    if x in {"도시공원", "도시공원놀이터"}:
        return "도시공원 놀이터"

    if x in {
        "교통관리", "교통단속", "차량번호인식", "주차인식", "차번", "차번동영상",
        "불법주정차 단속", "둔치주차장 관제"
    }:
        return "교통관리"

    if x in {
        "재난안전", "재난재해", "재난재해(산불)", "재난재해(산불감시)",
        "재난재해(도로 결빙)", "재난재해(도로결빙)", "재난재해(침수)",
        "재난재해(터널)", "재난재해(하천)", "재난재해(하천감시)",
        "재난재해(둔치주차장 관제 등)"
    }:
        return "재난재해"

    if x in {
        "쓰레기단속", "쓰레기투기", "쓰레기투기 감시", "쓰레기투기감시",
        "불법쓰레기투기", "불법투기 방지", "불법투기감시", "무단투기단속"
    }:
        return "쓰레기 투기감시"

    if x in {
        "시설물관리", "시설물 관리", "시설물관리(주차장)",
        "시설물보호 및 안전사고 예방", "시설안전 및 화재 예방용",
        "안전사고 예방", "안전사고 예방 및 시설물 보호",
        "청사방호", "청사 및 민원인 방호", "동 청사 방호 등",
        "주민자치센터 안전관리", "민원실 안전한 환경조성",
        "문화재관리", "문화회관 보안용", "동 주민센터 공용주차장 관리",
        "주차장 사고 예방"
    }:
        return "청사, 시설, 문화재관리"

    if x in {"기타", "다목적"}:
        return np.nan

    return np.nan

##### 3-0-4. CCTV 원본 의미 기준 칼럼 후보

In [ ]:
CCTV_FIELD_CANDIDATES = {
    "소재지": [
        "카메라 설치위치",
        "카메라 설치위치 ",
        "설치장소(도로명주소)",
        "소새지도로명주소",
        "소재지도로명주소",
        "소재지지번주소",
        "주소",
    ],
    "설치목적구분": [
        "설치목적구분", "설치목적", "설치 목적", "목적", "용도"
    ],
    "카메라수": [
        "카메라수", "카메라대수", "카메라 대수"
    ],
    "설치연도": [
        "설치연도", "설치년도", "설치연월", "설치년월", "설치일자", "설치일", "설치년"
    ],
    "위도": [
        "위도", "WGS84위도", "Latitude", "latitude", "lat"
    ],
    "경도": [
        "경도", "WGS84경도", "Longitude", "longitude", "lon", "lng"
    ],
}

def coalesce_series(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    found = [c for c in candidates if c in df.columns]
    if not found:
        return pd.Series([pd.NA] * len(df), index=df.index, dtype="object")

    out = df[found[0]].copy()
    for c in found[1:]:
        out = out.where(out.notna() & (out.astype("string").str.strip() != ""), df[c])
    return out

##### 3-0-5. 카카오 주소검색 API

In [ ]:
KAKAO_REST_API_KEY = os.getenv("KAKAO_REST_API_KEY", "")
KAKAO_REST_API_KEY = "921434ce7718ceab3b0559f42b028ef1"

KAKAO_ADDRESS_URL = "https://dapi.kakao.com/v2/local/search/address.json"

def build_kakao_query(gu: str, address_value: str) -> str | None:
    if pd.isna(address_value):
        return None

    address_value = str(address_value).strip()
    if address_value == "":
        return None

    if "대구" in address_value:
        return address_value

    return f"대구광역시 {gu} {address_value}"


def kakao_search_address_only(query: str, session: requests.Session, rest_api_key: str, sleep_sec: float = 0.03) -> dict:
    if pd.isna(query) or str(query).strip() == "":
        return {
            "success": False,
            "x": np.nan,
            "y": np.nan,
            "address_name": None,
            "address_type": None,
            "search_method": "주소검색",
            "message": "query_is_empty"
        }

    if not rest_api_key:
        raise ValueError("KAKAO_REST_API_KEY가 설정되지 않았습니다.")

    headers = {
        "Authorization": f"KakaoAK {rest_api_key}"
    }

    allowed_types = {"ROAD_ADDR", "REGION_ADDR"}

    for analyze_type in ["exact", "similar"]:
        try:
            resp = session.get(
                KAKAO_ADDRESS_URL,
                headers=headers,
                params={
                    "query": str(query).strip(),
                    "analyze_type": analyze_type
                },
                timeout=10
            )
            resp.raise_for_status()
            data = resp.json()

            docs = data.get("documents", [])
            filtered = [doc for doc in docs if doc.get("address_type") in allowed_types]

            if len(filtered) > 0:
                doc = filtered[0]
                time.sleep(sleep_sec)
                return {
                    "success": True,
                    "x": float(doc["x"]) if doc.get("x") not in [None, ""] else np.nan,
                    "y": float(doc["y"]) if doc.get("y") not in [None, ""] else np.nan,
                    "address_name": doc.get("address_name"),
                    "address_type": doc.get("address_type"),
                    "search_method": "주소검색",
                    "message": f"ok_{analyze_type}"
                }

        except requests.HTTPError as e:
            return {
                "success": False,
                "x": np.nan,
                "y": np.nan,
                "address_name": None,
                "address_type": None,
                "search_method": "주소검색",
                "message": f"http_error: {e}"
            }
        except Exception as e:
            return {
                "success": False,
                "x": np.nan,
                "y": np.nan,
                "address_name": None,
                "address_type": None,
                "search_method": "주소검색",
                "message": f"request_error: {e}"
            }

    return {
        "success": False,
        "x": np.nan,
        "y": np.nan,
        "address_name": None,
        "address_type": None,
        "search_method": "주소검색",
        "message": "no_valid_address_result"
    }

##### 3-0-6. geometry 검증용 함수

In [ ]:
def spatial_validate_chunked(df: pd.DataFrame, lat_col: str, lon_col: str, validate_gdf: gpd.GeoDataFrame, chunk_size: int = 5000, desc: str = "좌표검증") -> pd.DataFrame:
    result = pd.DataFrame(index=df.index)
    result["검증자치구"] = pd.NA
    result["검증성공여부"] = 0
    result["검증메시지"] = "coord_missing"

    valid_idx = df.index[df[lat_col].notna() & df[lon_col].notna()]

    if len(valid_idx) == 0:
        return result

    idx_list = list(valid_idx)

    for start in tqdm(range(0, len(idx_list), chunk_size), desc=desc, unit="chunk"):
        chunk_idx = idx_list[start:start + chunk_size]
        chunk = df.loc[chunk_idx, ["자치구", lat_col, lon_col]].copy()
        chunk = chunk.rename(columns={lat_col: "위도", lon_col: "경도"})

        point_gdf = gpd.GeoDataFrame(
            chunk[["자치구"]].copy(),
            geometry=gpd.points_from_xy(chunk["경도"], chunk["위도"]),
            crs="EPSG:4326"
        ).to_crs(validate_gdf.crs)

        joined = gpd.sjoin(
            point_gdf,
            validate_gdf[["자치구", "geometry"]].rename(columns={"자치구": "경계판정자치구"}),
            how="left",
            predicate="intersects"
        ).reset_index().rename(columns={"index": "원본인덱스"})

        joined["자치구"] = joined["자치구"].apply(standardize_gu_name)
        joined["경계판정자치구"] = joined["경계판정자치구"].apply(standardize_gu_name)

        joined["우선순위"] = (joined["자치구"] == joined["경계판정자치구"]).astype(int)
        joined = (
            joined
            .sort_values(["원본인덱스", "우선순위"], ascending=[True, False])
            .drop_duplicates(subset=["원본인덱스"], keep="first")
        )

        for _, row in joined.iterrows():
            idx = row["원본인덱스"]
            judged = row["경계판정자치구"]
            origin_gu = row["자치구"]

            result.at[idx, "검증자치구"] = judged

            if pd.isna(judged):
                result.at[idx, "검증성공여부"] = 0
                result.at[idx, "검증메시지"] = "outside_daegu_gu"
            elif judged == origin_gu:
                result.at[idx, "검증성공여부"] = 1
                result.at[idx, "검증메시지"] = "inside_same_gu"
            else:
                result.at[idx, "검증성공여부"] = 0
                result.at[idx, "검증메시지"] = f"inside_other_gu:{judged}"

    return result

### 3-1. 인구 데이터 전처리

In [ ]:
population_clean_dict = {}

for year, df in population_dfs.items():
    temp = df.copy()

    # 자치구명 추출 및 표준화
    temp["자치구"] = temp["행정구역"].apply(extract_admin_name).apply(standardize_gu_name)

    # 대구 전체 행 제거
    temp = temp[temp["자치구"] != "대구광역시"].copy()

    # 해당 연도 월별 총인구수 컬럼 추출
    pop_cols = [col for col in temp.columns if re.match(rf"^{year}년\d{{2}}월_총인구수$", col)]

    # 숫자형 변환
    for col in pop_cols:
        temp[col] = clean_numeric_series(temp[col])

    # 연평균 인구
    temp["연평균인구"] = temp[pop_cols].mean(axis=1)

    # 최소 결과물
    out = temp[["자치구", "연평균인구"]].copy()
    out["연도"] = year

    # 정렬
    out = out[["연도", "자치구", "연평균인구"]].sort_values(["연도", "자치구"]).reset_index(drop=True)

    population_clean_dict[year] = out

# 통합
population_clean_df = pd.concat(population_clean_dict.values(), ignore_index=True)

# 장기 비교용(군위군 제외)
population_long_df = population_clean_df[population_clean_df["자치구"].isin(DAEGU_GU_LIST_LONG)].copy()

# 2024 핵심 분석용(군위군 포함)
population_2024_df = population_clean_df[population_clean_df["연도"] == 2024].copy()

print("[population_clean_df]")
display(population_clean_df.head(20))

### 3-2. 범죄 데이터 전처리

In [ ]:
crime_clean_dict = {}
crime_total_dict = {}
crime_five_dict = {}
crime_five_pivot_dict = {}

for year, df in crime_dfs.items():
    raw = df.copy()

    # 현재 구조:
    # columns = ['범죄별(1)', '범죄별(2)', '범죄별(3)', '2024', '2024.1', ...]
    # raw.iloc[1, 3:] = 실제 자치구명
    base_cols = list(raw.columns[:3])
    region_cols = list(raw.columns[3:])

    region_names = raw.iloc[1, 3:].astype(str).str.strip().apply(standardize_gu_name).tolist()

    # 데이터 본문
    temp = raw.iloc[2:].copy().reset_index(drop=True)
    temp.columns = base_cols + region_names

    # 컬럼명 정리
    temp = temp.rename(columns={
        base_cols[0]: "범죄별1",
        base_cols[1]: "범죄별2",
        base_cols[2]: "범죄별3"
    })

    # long 변환
    long_df = temp.melt(
        id_vars=["범죄별1", "범죄별2", "범죄별3"],
        value_vars=region_names,
        var_name="자치구",
        value_name="범죄건수"
    )

    long_df["자치구"] = long_df["자치구"].apply(standardize_gu_name)
    long_df["범죄건수"] = clean_numeric_series(long_df["범죄건수"]).fillna(0)
    long_df["연도"] = year

    # 저장
    crime_clean_dict[year] = long_df.copy()

    # 전체 범죄 합계
    total_df = (
        long_df
        .groupby(["연도", "자치구"], as_index=False)["범죄건수"]
        .sum()
        .rename(columns={"범죄건수": "전체범죄건수"})
    )
    crime_total_dict[year] = total_df

    # 5대범죄 재분류
    five_df = long_df.copy()
    five_df["5대범죄구분"] = five_df["범죄별3"].apply(map_five_major_crime)
    five_df = five_df[five_df["5대범죄구분"].notna()].copy()

    # 5대범죄 유형별
    five_pivot = (
        five_df.groupby(["연도", "자치구", "5대범죄구분"], as_index=False)["범죄건수"]
        .sum()
        .pivot(index=["연도", "자치구"], columns="5대범죄구분", values="범죄건수")
        .reset_index()
        .fillna(0)
    )

    # 5대범죄 합계
    value_cols = [c for c in five_pivot.columns if c not in ["연도", "자치구"]]
    five_pivot["5대범죄건수"] = five_pivot[value_cols].sum(axis=1)

    crime_five_dict[year] = five_df
    crime_five_pivot_dict[year] = five_pivot

crime_clean_df = pd.concat(crime_clean_dict.values(), ignore_index=True)
crime_total_df = pd.concat(crime_total_dict.values(), ignore_index=True)
crime_five_pivot_df = pd.concat(crime_five_pivot_dict.values(), ignore_index=True)

# 장기 비교용(군위군 제외)
crime_total_long_df = crime_total_df[crime_total_df["자치구"].isin(DAEGU_GU_LIST_LONG)].copy()
crime_five_long_df = crime_five_pivot_df[crime_five_pivot_df["자치구"].isin(DAEGU_GU_LIST_LONG)].copy()

# 2024 핵심 분석용
crime_total_2024_df = crime_total_df[crime_total_df["연도"] == 2024].copy()
crime_five_2024_df = crime_five_pivot_df[crime_five_pivot_df["연도"] == 2024].copy()

print("[crime_clean_df]")
display(crime_clean_df.head(20))

print("\n[crime_total_df]")
display(crime_total_df.head(20))

print("\n[crime_five_pivot_df]")
display(crime_five_pivot_df.head(20))

### 3-3. 경계 데이터 전처리 및 2024 남구 geometry 교체

In [ ]:
geo_clean_dict = {}

for year, gdf in geo_dfs.items():
    temp = gdf.copy()

    # 존재하는 컬럼만 문자열 정리
    existing_cols = [col for col in ["A1", "A2", "A4"] if col in temp.columns]
    for col in existing_cols:
        temp[col] = temp[col].astype(str).str.strip()

    valid_codes = set(DAEGU_SIG_MAP.keys())

    # A4 없는 연도 대비
    if "A4" not in temp.columns:
        temp["A4"] = np.nan

    # 대구 자치구만 필터링
    temp = temp[
        temp["A1"].isin(valid_codes) | temp["A4"].isin(valid_codes)
    ].copy()

    # SIG 코드 확정
    temp["SIG_CD"] = np.where(
        temp["A1"].isin(valid_codes),
        temp["A1"],
        temp["A4"]
    )

    # 자치구명 표준화
    if "A2" not in temp.columns:
        raise ValueError(f"{year}년 경계 데이터에 A2 컬럼이 없습니다.")

    temp["자치구"] = temp["A2"].apply(standardize_gu_name)

    # 필요한 컬럼만 유지
    temp = temp[["SIG_CD", "자치구", "geometry"]].copy()

    # CRS 통일
    if temp.crs is None:
        raise ValueError(f"{year}년 경계 데이터 CRS가 없습니다.")

    if temp.crs.to_epsg() != 5186:
        temp = temp.to_crs(epsg=5186)

    # invalid geometry 보정
    invalid_mask = ~temp.geometry.is_valid
    if invalid_mask.any():
        temp.loc[invalid_mask, "geometry"] = temp.loc[invalid_mask, "geometry"].buffer(0)

    geo_clean_dict[year] = temp

# 2024 남구 geometry를 2023년 남구 geometry로 교체
geo_2023 = geo_clean_dict[2023].copy()
geo_2024 = geo_clean_dict[2024].copy()

namgu_geom_2023 = geo_2023.loc[geo_2023["SIG_CD"] == "27200", "geometry"].iloc[0]
geo_2024.loc[geo_2024["SIG_CD"] == "27200", "geometry"] = namgu_geom_2023

geo_clean_dict[2024] = geo_2024

# 면적 계산
for year in geo_clean_dict.keys():
    geo_clean_dict[year]["자치구총면적_km2"] = geo_clean_dict[year].geometry.area / 1_000_000

geo_clean_df = pd.concat(
    [g.assign(연도=year) for year, g in geo_clean_dict.items()],
    ignore_index=True
)[["연도", "SIG_CD", "자치구", "자치구총면적_km2", "geometry"]]

geo_long_df = geo_clean_df[geo_clean_df["자치구"].isin(DAEGU_GU_LIST_LONG)].copy()
geo_2024_df = geo_clean_df[geo_clean_df["연도"] == 2024].copy()

# 2024 검증용 경계
validate_gdf_2024 = geo_clean_dict[2024].copy()

# sanity check
expected_gu_2024 = set(DAEGU_GU_LIST)
actual_gu_2024 = set(geo_2024_df["자치구"].astype(str).tolist())

missing_gu = expected_gu_2024 - actual_gu_2024
if missing_gu:
    raise ValueError(f"2024 경계 데이터에 누락된 자치구가 있습니다: {missing_gu}")

print("[geo_2024_df]")
display(geo_2024_df.sort_values("자치구").reset_index(drop=True))

### 3-4. CCTV 데이터 전처리 + 카카오 주소검색 후보 생성

In [ ]:
cctv_preclean_list = []

for gu, df in cctv_dfs.items():
    temp = df.copy()
    temp.columns = [str(c).strip() for c in temp.columns]
    temp["자치구"] = gu

    소재지_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["소재지"])
    설치목적_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["설치목적구분"])
    카메라수_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["카메라수"])
    설치연도_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["설치연도"])
    위도_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["위도"])
    경도_raw = coalesce_series(temp, CCTV_FIELD_CANDIDATES["경도"])

    temp["소재지"] = normalize_text_series(소재지_raw)
    temp["설치목적구분"] = normalize_text_series(설치목적_raw)
    temp["설치목적표준"] = temp["설치목적구분"].apply(map_cctv_purpose)
    temp["설치목적미분류여부"] = temp["설치목적표준"].isna().astype(int)

    temp["카메라수"] = clean_numeric_series(카메라수_raw)
    temp["카메라수결측대체여부"] = temp["카메라수"].isna().astype(int)
    temp["카메라수보정"] = temp["카메라수"].fillna(1)

    temp["설치연도"] = extract_install_year(설치연도_raw)
    temp["설치연도존재여부"] = temp["설치연도"].notna().astype(int)

    temp["원본위도"] = clean_numeric_series(위도_raw)
    temp["원본경도"] = clean_numeric_series(경도_raw)
    temp["원본좌표존재여부"] = (
        temp["원본위도"].notna() & temp["원본경도"].notna()
    ).astype(int)

    temp["지오코딩검색어"] = temp.apply(
        lambda r: build_kakao_query(r["자치구"], r["소재지"]),
        axis=1
    )

    cctv_preclean_list.append(
        temp[
            [
                "자치구",
                "소재지",
                "설치목적구분",
                "설치목적표준",
                "설치목적미분류여부",
                "카메라수",
                "카메라수보정",
                "카메라수결측대체여부",
                "설치연도",
                "설치연도존재여부",
                "원본위도",
                "원본경도",
                "원본좌표존재여부",
                "지오코딩검색어",
            ]
        ].copy()
    )

cctv_preclean_df = pd.concat(cctv_preclean_list, ignore_index=True)

need_geocode_mask = (
    (cctv_preclean_df["원본좌표존재여부"] == 0) &
    (cctv_preclean_df["지오코딩검색어"].notna())
)

unique_queries = (
    cctv_preclean_df.loc[need_geocode_mask, "지오코딩검색어"]
    .astype(str)
    .str.strip()
    .dropna()
    .unique()
)

geocode_results = []
session = requests.Session()

for query in tqdm(unique_queries, desc="3-4 주소검색", unit="query"):
    result = kakao_search_address_only(query, session=session, rest_api_key=KAKAO_REST_API_KEY)
    geocode_results.append({
        "지오코딩검색어": query,
        "API위도": result["y"],
        "API경도": result["x"],
        "API주소명": result["address_name"],
        "API주소유형": result["address_type"],
        "좌표검색방식": result["search_method"],
        "지오코딩성공여부": int(result["success"]),
        "지오코딩메시지": result["message"],
    })

geocode_result_df = pd.DataFrame(geocode_results)

cctv_preclean_df = cctv_preclean_df.merge(
    geocode_result_df,
    on="지오코딩검색어",
    how="left"
)

for col in ["API위도", "API경도", "API주소명", "API주소유형", "좌표검색방식", "지오코딩성공여부", "지오코딩메시지"]:
    if col not in cctv_preclean_df.columns:
        cctv_preclean_df[col] = pd.NA

cctv_preclean_df["지오코딩성공여부"] = cctv_preclean_df["지오코딩성공여부"].fillna(0).astype(int)

print("[cctv_preclean_df]")
display(cctv_preclean_df.head(20))

print("\n[cctv_preclean_df shape]")
print(cctv_preclean_df.shape)

print("\n[지오코딩 대상 요약]")
display(
    cctv_preclean_df.groupby("자치구", as_index=False).agg(
        전체행수=("자치구", "size"),
        원본좌표행수=("원본좌표존재여부", "sum"),
        지오코딩성공행수=("지오코딩성공여부", "sum"),
    )
)

### 3-5. geometry 기준 좌표 검증 및 최종 좌표 확정

In [ ]:
origin_valid_df = spatial_validate_chunked(
    cctv_preclean_df,
    lat_col="원본위도",
    lon_col="원본경도",
    validate_gdf=validate_gdf_2024,
    chunk_size=5000,
    desc="3-5 원본좌표 검증"
)

api_valid_df = spatial_validate_chunked(
    cctv_preclean_df,
    lat_col="API위도",
    lon_col="API경도",
    validate_gdf=validate_gdf_2024,
    chunk_size=5000,
    desc="3-5 API좌표 검증"
)

cctv_clean_df = cctv_preclean_df.copy()

cctv_clean_df["원본검증자치구"] = origin_valid_df["검증자치구"]
cctv_clean_df["원본좌표유효여부"] = origin_valid_df["검증성공여부"]
cctv_clean_df["원본좌표검증메시지"] = origin_valid_df["검증메시지"]

cctv_clean_df["API검증자치구"] = api_valid_df["검증자치구"]
cctv_clean_df["API좌표유효여부"] = api_valid_df["검증성공여부"]
cctv_clean_df["API좌표검증메시지"] = api_valid_df["검증메시지"]

cctv_clean_df["위도"] = np.nan
cctv_clean_df["경도"] = np.nan
cctv_clean_df["좌표출처"] = "없음"
cctv_clean_df["좌표유효여부"] = 0
cctv_clean_df["좌표검증메시지"] = pd.NA

use_origin_idx = cctv_clean_df.index[cctv_clean_df["원본좌표유효여부"] == 1]
cctv_clean_df.loc[use_origin_idx, "위도"] = cctv_clean_df.loc[use_origin_idx, "원본위도"]
cctv_clean_df.loc[use_origin_idx, "경도"] = cctv_clean_df.loc[use_origin_idx, "원본경도"]
cctv_clean_df.loc[use_origin_idx, "좌표출처"] = "원본"
cctv_clean_df.loc[use_origin_idx, "좌표유효여부"] = 1
cctv_clean_df.loc[use_origin_idx, "좌표검증메시지"] = cctv_clean_df.loc[use_origin_idx, "원본좌표검증메시지"]

use_api_idx = cctv_clean_df.index[
    (cctv_clean_df["원본좌표유효여부"] == 0) &
    (cctv_clean_df["API좌표유효여부"] == 1)
]
cctv_clean_df.loc[use_api_idx, "위도"] = cctv_clean_df.loc[use_api_idx, "API위도"]
cctv_clean_df.loc[use_api_idx, "경도"] = cctv_clean_df.loc[use_api_idx, "API경도"]
cctv_clean_df.loc[use_api_idx, "좌표출처"] = "카카오API"
cctv_clean_df.loc[use_api_idx, "좌표유효여부"] = 1
cctv_clean_df.loc[use_api_idx, "좌표검증메시지"] = cctv_clean_df.loc[use_api_idx, "API좌표검증메시지"]

cctv_clean_df["좌표검색방식"] = np.where(
    cctv_clean_df["좌표출처"] == "카카오API",
    "주소검색",
    pd.NA
)

cctv_clean_df = cctv_clean_df[
    [
        "자치구",
        "소재지",
        "설치목적구분",
        "설치목적표준",
        "설치목적미분류여부",
        "카메라수",
        "카메라수보정",
        "카메라수결측대체여부",
        "설치연도",
        "설치연도존재여부",
        "위도",
        "경도",
        "좌표출처",
        "좌표검색방식",
        "지오코딩검색어",
        "지오코딩성공여부",
        "지오코딩메시지",
        "좌표유효여부",
        "좌표검증메시지",
        "원본위도",
        "원본경도",
        "원본좌표유효여부",
        "API위도",
        "API경도",
        "API좌표유효여부",
    ]
].copy()

print("[cctv_clean_df]")
display(cctv_clean_df.head(20))

print("\n[cctv_clean_df shape]")
print(cctv_clean_df.shape)

coord_summary_df = (
    cctv_clean_df
    .groupby("자치구", as_index=False)
    .agg(
        전체행수=("자치구", "size"),
        원본좌표채택행수=("좌표출처", lambda x: (x == "원본").sum()),
        카카오좌표채택행수=("좌표출처", lambda x: (x == "카카오API").sum()),
        좌표없음행수=("좌표출처", lambda x: (x == "없음").sum()),
        좌표유효행수=("좌표유효여부", "sum"),
    )
)
coord_summary_df["최종좌표존재행수"] = coord_summary_df["원본좌표채택행수"] + coord_summary_df["카카오좌표채택행수"]
coord_summary_df["최종좌표존재비율"] = (
    coord_summary_df["최종좌표존재행수"] / coord_summary_df["전체행수"]
).round(4)

print("\n[자치구별 최종 좌표 요약]")
display(coord_summary_df.sort_values("자치구").reset_index(drop=True))

### 3-6. 용도지역 데이터 전처리

In [ ]:
landuse_long_list = []
landuse_summary_list = []

for year, df in landuse_dfs.items():
    temp = df.copy()

    # 자치구 면적 컬럼만 추출(대구 전체 제외)
    area_cols = [col for col in temp.columns if col.endswith("(m2)") and col != "대구(m2)"]

    # long 변환
    long_df = temp.melt(
        id_vars=["구분", "상세구분"],
        value_vars=area_cols,
        var_name="자치구_원본",
        value_name="면적_m2"
    )

    long_df["자치구"] = (
        long_df["자치구_원본"]
        .str.replace("(m2)", "", regex=False)
        .str.strip()
        .apply(standardize_gu_name)
    )

    long_df["면적_m2"] = clean_numeric_series(long_df["면적_m2"]).fillna(0)
    long_df["면적_km2"] = long_df["면적_m2"] / 1_000_000
    long_df["연도"] = year

    landuse_long_list.append(long_df[["연도", "자치구", "구분", "상세구분", "면적_m2", "면적_km2"]])

    # 상위 구분 기준 요약
    summary = (
        long_df.groupby(["연도", "자치구", "구분"], as_index=False)["면적_km2"]
        .sum()
        .pivot(index=["연도", "자치구"], columns="구분", values="면적_km2")
        .reset_index()
        .fillna(0)
    )

    # 핵심 변수명 정리
    rename_map = {}
    if "주거지역" in summary.columns:
        rename_map["주거지역"] = "주거지역면적_km2"
    if "상업지역" in summary.columns:
        rename_map["상업지역"] = "상업지역면적_km2"
    if "공업지역" in summary.columns:
        rename_map["공업지역"] = "공업지역면적_km2"

    summary = summary.rename(columns=rename_map)

    # 없으면 0 컬럼 생성
    for col in ["주거지역면적_km2", "상업지역면적_km2", "공업지역면적_km2"]:
        if col not in summary.columns:
            summary[col] = 0

    # 용도지역 총합(검증용)
    land_cols = [c for c in summary.columns if c not in ["연도", "자치구"]]
    summary["용도지역총합_km2"] = summary[land_cols].sum(axis=1)

    landuse_summary_list.append(summary)

landuse_long_df = pd.concat(landuse_long_list, ignore_index=True)
landuse_summary_df = pd.concat(landuse_summary_list, ignore_index=True)

# 장기 비교용(군위군 제외)
landuse_long_compare_df = landuse_summary_df[landuse_summary_df["자치구"].isin(DAEGU_GU_LIST_LONG)].copy()

# 2024 핵심 분석용
landuse_2024_summary_df = landuse_summary_df[landuse_summary_df["연도"] == 2024].copy()

print("[landuse_long_df]")
display(landuse_long_df.head(20))

print("\n[landuse_summary_df]")
display(landuse_summary_df.head(20))

### 3-7. CCTV 자치구 단위 집계

In [ ]:
cctv_purpose_summary = (
    cctv_clean_df
    .groupby(["자치구", "설치목적표준"], dropna=False, as_index=False)["카메라수보정"]
    .sum()
)

cctv_purpose_summary["설치목적표준"] = cctv_purpose_summary["설치목적표준"].fillna("미분류")

cctv_purpose_wide = (
    cctv_purpose_summary
    .pivot(index="자치구", columns="설치목적표준", values="카메라수보정")
    .reset_index()
    .fillna(0)
)

cctv_clean_df["좌표유효CCTV수_기여값"] = cctv_clean_df["카메라수보정"] * cctv_clean_df["좌표유효여부"]
cctv_clean_df["원본좌표채택CCTV수_기여값"] = cctv_clean_df["카메라수보정"] * (cctv_clean_df["좌표출처"] == "원본").astype(int)
cctv_clean_df["카카오좌표채택CCTV수_기여값"] = cctv_clean_df["카메라수보정"] * (cctv_clean_df["좌표출처"] == "카카오API").astype(int)

cctv_summary_df = (
    cctv_clean_df
    .groupby("자치구", as_index=False)
    .agg(
        전체행수=("자치구", "size"),
        CCTV총량=("카메라수보정", "sum"),

        좌표유효행수=("좌표유효여부", "sum"),
        좌표유효CCTV수=("좌표유효CCTV수_기여값", "sum"),

        원본좌표채택행수=("좌표출처", lambda x: (x == "원본").sum()),
        카카오좌표채택행수=("좌표출처", lambda x: (x == "카카오API").sum()),

        원본좌표채택CCTV수=("원본좌표채택CCTV수_기여값", "sum"),
        카카오좌표채택CCTV수=("카카오좌표채택CCTV수_기여값", "sum"),

        설치연도존재행수=("설치연도존재여부", "sum"),
        설치목적미분류행수=("설치목적미분류여부", "sum"),
        카메라수결측대체행수=("카메라수결측대체여부", "sum"),
    )
)

# -----------------------------
# 비율 계산 수정
# -----------------------------
# 전체 행 중 좌표가 최종 유효한 행 비율
cctv_summary_df["좌표유효행비율"] = (
    cctv_summary_df["좌표유효행수"] /
    cctv_summary_df["전체행수"].replace(0, np.nan)
).round(4)

# 전체 CCTV 총량 중 좌표가 유효한 CCTV 수 비율
cctv_summary_df["좌표유효CCTV비율"] = (
    cctv_summary_df["좌표유효CCTV수"] /
    cctv_summary_df["CCTV총량"].replace(0, np.nan)
).round(4)

# 전체 행 중 카메라수 결측을 1로 대체한 행 비율
cctv_summary_df["카메라수결측대체비율"] = (
    cctv_summary_df["카메라수결측대체행수"] /
    cctv_summary_df["전체행수"].replace(0, np.nan)
).round(4)

# 목적별 수 merge
cctv_summary_df = (
    cctv_summary_df
    .merge(cctv_purpose_wide, on="자치구", how="left")
    .fillna(0)
)

purpose_cols = [
    "방범",
    "어린이 보호구역",
    "도시공원 놀이터",
    "교통관리",
    "재난재해",
    "쓰레기 투기감시",
    "청사, 시설, 문화재관리",
    "미분류",
]

for col in purpose_cols:
    if col not in cctv_summary_df.columns:
        cctv_summary_df[col] = 0

print("[cctv_summary_df]")
display(cctv_summary_df.sort_values("자치구").reset_index(drop=True))

### 3-8. 2024 핵심 분석용 통합 전처리 결과물 생성

In [ ]:
analysis_2024_df = (
    population_2024_df[["자치구", "연평균인구"]]
    .merge(crime_total_2024_df[["자치구", "전체범죄건수"]], on="자치구", how="left")
    .merge(crime_five_2024_df, on="자치구", how="left")
    .merge(cctv_summary_df, on="자치구", how="left")
    .merge(landuse_2024_summary_df.drop(columns=["연도"]), on="자치구", how="left")
    .merge(geo_2024_df[["자치구", "자치구총면적_km2"]], on="자치구", how="left")
)

analysis_2024_df["자치구"] = analysis_2024_df["자치구"].astype(str).str.strip().apply(standardize_gu_name)

missing_area_gu = analysis_2024_df.loc[
    analysis_2024_df["자치구총면적_km2"].isna(),
    "자치구"
].tolist()

if missing_area_gu:
    raise ValueError(f"2024 분석 테이블에서 총면적이 누락된 자치구가 있습니다: {missing_area_gu}")

for col in analysis_2024_df.columns:
    if col not in ["자치구", "자치구총면적_km2"]:
        analysis_2024_df[col] = analysis_2024_df[col].fillna(0)

print("[analysis_2024_df]")
display(analysis_2024_df.sort_values("자치구").reset_index(drop=True))

### 3-9. 2019~2024 장기 비교용(군위군 제외) 전처리 결과물 생성

In [ ]:
analysis_long_df = (
    population_long_df
    .merge(crime_total_long_df, on=["연도", "자치구"], how="left")
    .merge(crime_five_long_df, on=["연도", "자치구"], how="left")
    .merge(landuse_long_compare_df, on=["연도", "자치구"], how="left")
    .merge(
        geo_long_df[["연도", "자치구", "자치구총면적_km2"]],
        on=["연도", "자치구"],
        how="left"
    )
)

missing_area_long = analysis_long_df.loc[
    analysis_long_df["자치구총면적_km2"].isna(),
    ["연도", "자치구"]
]

if len(missing_area_long) > 0:
    raise ValueError(
        "장기 비교 테이블에서 총면적이 누락된 행이 있습니다.\n"
        f"{missing_area_long.to_dict(orient='records')}"
    )

for col in analysis_long_df.columns:
    if col not in ["연도", "자치구", "자치구총면적_km2"]:
        analysis_long_df[col] = analysis_long_df[col].fillna(0)

print("[analysis_long_df]")
display(analysis_long_df.head(30))

### 3-10. 전처리 결과 저장

In [ ]:
camera_assumption_df = (
    cctv_clean_df
    .groupby("자치구", as_index=False)
    .agg(
        전체행수=("자치구", "size"),
        카메라수결측대체행수=("카메라수결측대체여부", "sum"),
        CCTV총량_보정합=("카메라수보정", "sum")
    )
)

camera_assumption_df["카메라수결측대체비율"] = (
    camera_assumption_df["카메라수결측대체행수"] /
    camera_assumption_df["전체행수"]
).round(4)

# 인구
population_clean_df.to_csv(os.path.join(output_path, "population_clean.csv"), index=False, encoding="utf-8-sig")
population_long_df.to_csv(os.path.join(output_path, "population_long.csv"), index=False, encoding="utf-8-sig")
population_2024_df.to_csv(os.path.join(output_path, "population_2024.csv"), index=False, encoding="utf-8-sig")

# 범죄
crime_clean_df.to_csv(os.path.join(output_path, "crime_clean_long.csv"), index=False, encoding="utf-8-sig")
crime_total_df.to_csv(os.path.join(output_path, "crime_total_by_gu_year.csv"), index=False, encoding="utf-8-sig")
crime_five_pivot_df.to_csv(os.path.join(output_path, "crime_five_by_gu_year.csv"), index=False, encoding="utf-8-sig")
crime_total_2024_df.to_csv(os.path.join(output_path, "crime_total_2024.csv"), index=False, encoding="utf-8-sig")
crime_five_2024_df.to_csv(os.path.join(output_path, "crime_five_2024.csv"), index=False, encoding="utf-8-sig")

# CCTV
cctv_preclean_df.to_csv(os.path.join(output_path, "cctv_preclean.csv"), index=False, encoding="utf-8-sig")
cctv_clean_df.to_csv(os.path.join(output_path, "cctv_clean.csv"), index=False, encoding="utf-8-sig")
cctv_summary_df.to_csv(os.path.join(output_path, "cctv_summary_by_gu.csv"), index=False, encoding="utf-8-sig")
coord_summary_df.to_csv(os.path.join(output_path, "cctv_coord_summary_by_gu.csv"), index=False, encoding="utf-8-sig")
camera_assumption_df.to_csv(os.path.join(output_path, "cctv_camera_assumption_by_gu.csv"), index=False, encoding="utf-8-sig")

# 용도지역
landuse_long_df.to_csv(os.path.join(output_path, "landuse_long.csv"), index=False, encoding="utf-8-sig")
landuse_summary_df.to_csv(os.path.join(output_path, "landuse_summary_by_gu_year.csv"), index=False, encoding="utf-8-sig")
landuse_2024_summary_df.to_csv(os.path.join(output_path, "landuse_summary_2024.csv"), index=False, encoding="utf-8-sig")

# 경계
geo_clean_df.drop(columns="geometry").to_csv(os.path.join(output_path, "geo_area_by_gu_year.csv"), index=False, encoding="utf-8-sig")
geo_2024_df.drop(columns="geometry").to_csv(os.path.join(output_path, "geo_area_2024.csv"), index=False, encoding="utf-8-sig")
geo_long_df.drop(columns="geometry").to_csv(os.path.join(output_path, "geo_area_long.csv"), index=False, encoding="utf-8-sig")

# 최종 통합 분석 테이블
analysis_2024_df.to_csv(os.path.join(output_path, "analysis_2024_preprocessed.csv"), index=False, encoding="utf-8-sig")
analysis_long_df.to_csv(os.path.join(output_path, "analysis_long_preprocessed.csv"), index=False, encoding="utf-8-sig")

print("전처리 결과 저장 완료")

# 4. 파생변수 생성

### 4-0. 파생변수 생성용 함수

In [ ]:
def safe_divide(numerator, denominator):
    """
    0으로 나누는 경우를 방지하는 안전한 나눗셈
    denominator가 0 또는 결측이면 NaN 반환
    """
    num = pd.to_numeric(numerator, errors="coerce")
    den = pd.to_numeric(denominator, errors="coerce")

    result = np.where((den.notna()) & (den != 0), num / den, np.nan)
    return result

### 4-1. 2024 단면 분석용 파생변수 생성

In [ ]:
analysis_2024_derived_df = analysis_2024_df.copy()

##### 4-1-1. 범죄 관련 파생 변수

In [ ]:
analysis_2024_derived_df["인구10만명당_전체범죄율"] = (
    safe_divide(analysis_2024_derived_df["전체범죄건수"], analysis_2024_derived_df["연평균인구"]) * 100000
)

analysis_2024_derived_df["인구10만명당_5대범죄율"] = (
    safe_divide(analysis_2024_derived_df["5대범죄건수"], analysis_2024_derived_df["연평균인구"]) * 100000
)

analysis_2024_derived_df["5대범죄비중"] = safe_divide(
    analysis_2024_derived_df["5대범죄건수"],
    analysis_2024_derived_df["전체범죄건수"]
)

if "폭력" in analysis_2024_derived_df.columns:
    analysis_2024_derived_df["폭력비중"] = safe_divide(
        analysis_2024_derived_df["폭력"],
        analysis_2024_derived_df["5대범죄건수"]
    )
else:
    analysis_2024_derived_df["폭력비중"] = np.nan

4-1-2. CCTV 관련 파생 변수

In [ ]:
analysis_2024_derived_df["인구10만명당_CCTV수"] = (
    safe_divide(analysis_2024_derived_df["CCTV총량"], analysis_2024_derived_df["연평균인구"]) * 100000
)

analysis_2024_derived_df["km2당_CCTV수"] = safe_divide(
    analysis_2024_derived_df["CCTV총량"],
    analysis_2024_derived_df["자치구총면적_km2"]
)

analysis_2024_derived_df["범죄1건당_CCTV수"] = safe_divide(
    analysis_2024_derived_df["CCTV총량"],
    analysis_2024_derived_df["전체범죄건수"]
)

analysis_2024_derived_df["5대범죄1건당_CCTV수"] = safe_divide(
    analysis_2024_derived_df["CCTV총량"],
    analysis_2024_derived_df["5대범죄건수"]
)

if "좌표유효CCTV수" in analysis_2024_derived_df.columns:
    analysis_2024_derived_df["좌표유효CCTV비율"] = safe_divide(
        analysis_2024_derived_df["좌표유효CCTV수"],
        analysis_2024_derived_df["CCTV총량"]
    )
else:
    analysis_2024_derived_df["좌표유효CCTV비율"] = np.nan

##### 4-1-3. CCTV 설치목적 비중

In [ ]:
purpose_cols = [
    "교통관리",
    "도시공원 놀이터",
    "미분류",
    "방범",
    "쓰레기 투기감시",
    "어린이 보호구역",
    "재난재해",
    "청사, 시설, 문화재관리"
]

for col in purpose_cols:
    if col not in analysis_2024_derived_df.columns:
        analysis_2024_derived_df[col] = 0

    analysis_2024_derived_df[f"{col}_비중"] = safe_divide(
        analysis_2024_derived_df[col],
        analysis_2024_derived_df["CCTV총량"]
    )

analysis_2024_derived_df["방범CCTV비중"] = analysis_2024_derived_df["방범_비중"]
analysis_2024_derived_df["교통관리CCTV비중"] = analysis_2024_derived_df["교통관리_비중"]
analysis_2024_derived_df["재난재해CCTV비중"] = analysis_2024_derived_df["재난재해_비중"]
analysis_2024_derived_df["어린이보호구역CCTV비중"] = analysis_2024_derived_df["어린이 보호구역_비중"]
analysis_2024_derived_df["청사시설문화재관리CCTV비중"] = analysis_2024_derived_df["청사, 시설, 문화재관리_비중"]
analysis_2024_derived_df["미분류CCTV비중"] = analysis_2024_derived_df["미분류_비중"]

##### 4-1-4. 용도지역 관련 파생변수

In [ ]:
analysis_2024_derived_df["주거지역비율"] = safe_divide(
    analysis_2024_derived_df["주거지역면적_km2"],
    analysis_2024_derived_df["자치구총면적_km2"]
)

analysis_2024_derived_df["상업지역비율"] = safe_divide(
    analysis_2024_derived_df["상업지역면적_km2"],
    analysis_2024_derived_df["자치구총면적_km2"]
)

analysis_2024_derived_df["공업지역비율"] = safe_divide(
    analysis_2024_derived_df["공업지역면적_km2"],
    analysis_2024_derived_df["자치구총면적_km2"]
)

##### 4-1-5. 검증용 보조 변수

In [ ]:
analysis_2024_derived_df["용도지역면적_총면적비율"] = safe_divide(
    analysis_2024_derived_df["용도지역총합_km2"],
    analysis_2024_derived_df["자치구총면적_km2"]
)

##### 4-1-6. 결과

In [ ]:
print("[analysis_2024_derived_df]")
display(analysis_2024_derived_df.sort_values("자치구").reset_index(drop=True))

### 4-2. 2019~2024 장기 비교용 파생변수 생성

In [ ]:
analysis_long_derived_df = analysis_long_df.copy()

##### 4-2-1. 범죄 관련 파생변수

In [ ]:
analysis_long_derived_df["인구10만명당_전체범죄율"] = (
    safe_divide(analysis_long_derived_df["전체범죄건수"], analysis_long_derived_df["연평균인구"]) * 100000
)

analysis_long_derived_df["인구10만명당_5대범죄율"] = (
    safe_divide(analysis_long_derived_df["5대범죄건수"], analysis_long_derived_df["연평균인구"]) * 100000
)

analysis_long_derived_df["5대범죄비중"] = safe_divide(
    analysis_long_derived_df["5대범죄건수"],
    analysis_long_derived_df["전체범죄건수"]
)

if "폭력" in analysis_long_derived_df.columns:
    analysis_long_derived_df["폭력비중"] = safe_divide(
        analysis_long_derived_df["폭력"],
        analysis_long_derived_df["5대범죄건수"]
    )
else:
    analysis_long_derived_df["폭력비중"] = np.nan

##### 4-2-2. 용도지역 관련 파생변수

In [ ]:
analysis_long_derived_df["주거지역비율"] = safe_divide(
    analysis_long_derived_df["주거지역면적_km2"],
    analysis_long_derived_df["자치구총면적_km2"]
)

analysis_long_derived_df["상업지역비율"] = safe_divide(
    analysis_long_derived_df["상업지역면적_km2"],
    analysis_long_derived_df["자치구총면적_km2"]
)

analysis_long_derived_df["공업지역비율"] = safe_divide(
    analysis_long_derived_df["공업지역면적_km2"],
    analysis_long_derived_df["자치구총면적_km2"]
)

analysis_long_derived_df["용도지역면적_총면적비율"] = safe_divide(
    analysis_long_derived_df["용도지역총합_km2"],
    analysis_long_derived_df["자치구총면적_km2"]
)

##### 4-2-3. 결과

In [ ]:
print("[analysis_long_derived_df]")
display(analysis_long_derived_df.head(30))

### 4-3. 2024 핵심 분석용 칼럼과 검증용 칼럼 분리

In [ ]:
analysis_2024_core_cols = [
    "자치구",
    "연평균인구",
    "전체범죄건수",
    "5대범죄건수",
    "강도",
    "살인",
    "성범죄",
    "절도",
    "폭력",
    "CCTV총량",
    "좌표유효CCTV수",
    "방범",
    "교통관리",
    "도시공원 놀이터",
    "쓰레기 투기감시",
    "어린이 보호구역",
    "재난재해",
    "청사, 시설, 문화재관리",
    "미분류",
    "주거지역면적_km2",
    "상업지역면적_km2",
    "공업지역면적_km2",
    "자치구총면적_km2",
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "인구10만명당_CCTV수",
    "km2당_CCTV수",
    "범죄1건당_CCTV수",
    "5대범죄1건당_CCTV수",
    "5대범죄비중",
    "폭력비중",
    "방범CCTV비중",
    "교통관리CCTV비중",
    "재난재해CCTV비중",
    "어린이보호구역CCTV비중",
    "청사시설문화재관리CCTV비중",
    "미분류CCTV비중",
    "주거지역비율",
    "상업지역비율",
    "공업지역비율",
]

analysis_2024_check_cols = [
    "자치구",
    "좌표유효행수",
    "좌표유효CCTV수",
    "원본좌표채택행수",
    "카카오좌표채택행수",
    "원본좌표채택CCTV수",
    "카카오좌표채택CCTV수",
    "설치연도존재행수",
    "설치목적미분류행수",
    "카메라수결측대체행수",
    "좌표유효행비율",
    "좌표유효CCTV비율",
    "카메라수결측대체비율",
    "용도지역총합_km2",
    "용도지역면적_총면적비율",
]

analysis_2024_core_cols = [c for c in analysis_2024_core_cols if c in analysis_2024_derived_df.columns]
analysis_2024_check_cols = [c for c in analysis_2024_check_cols if c in analysis_2024_derived_df.columns]

analysis_2024_core_df = analysis_2024_derived_df[analysis_2024_core_cols].copy()
analysis_2024_check_df = analysis_2024_derived_df[analysis_2024_check_cols].copy()

print("[analysis_2024_core_df]")
display(analysis_2024_core_df.sort_values("자치구").reset_index(drop=True))

print("\n[analysis_2024_check_df]")
display(analysis_2024_check_df.sort_values("자치구").reset_index(drop=True))

### 4-4. 장기 비교용 핵심 분석 칼럼과 검증용 칼럼 분리

In [ ]:
analysis_long_core_cols = [
    "연도",
    "자치구",
    "연평균인구",
    "전체범죄건수",
    "5대범죄건수",
    "강도",
    "살인",
    "성범죄",
    "절도",
    "폭력",
    "주거지역면적_km2",
    "상업지역면적_km2",
    "공업지역면적_km2",
    "자치구총면적_km2",
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "5대범죄비중",
    "폭력비중",
    "주거지역비율",
    "상업지역비율",
    "공업지역비율",
]

analysis_long_check_cols = [
    "연도",
    "자치구",
    "용도지역총합_km2",
    "용도지역면적_총면적비율",
]

analysis_long_core_cols = [c for c in analysis_long_core_cols if c in analysis_long_derived_df.columns]
analysis_long_check_cols = [c for c in analysis_long_check_cols if c in analysis_long_derived_df.columns]

analysis_long_core_df = analysis_long_derived_df[analysis_long_core_cols].copy()
analysis_long_check_df = analysis_long_derived_df[analysis_long_check_cols].copy()

print("[analysis_long_core_df]")
display(analysis_long_core_df.head(30))

print("\n[analysis_long_check_df]")
display(analysis_long_check_df.head(30))

### 4-5. 파생변수 결과 저장

In [ ]:
analysis_2024_derived_df.to_csv(
    os.path.join(output_path, "analysis_2024_derived.csv"),
    index=False,
    encoding="utf-8-sig"
)

analysis_long_derived_df.to_csv(
    os.path.join(output_path, "analysis_long_derived.csv"),
    index=False,
    encoding="utf-8-sig"
)

analysis_2024_core_df.to_csv(
    os.path.join(output_path, "analysis_2024_core.csv"),
    index=False,
    encoding="utf-8-sig"
)

analysis_2024_check_df.to_csv(
    os.path.join(output_path, "analysis_2024_check.csv"),
    index=False,
    encoding="utf-8-sig"
)

analysis_long_core_df.to_csv(
    os.path.join(output_path, "analysis_long_core.csv"),
    index=False,
    encoding="utf-8-sig"
)

analysis_long_check_df.to_csv(
    os.path.join(output_path, "analysis_long_check.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("파생변수 결과 저장 완료")

# 5. EDA

### 5-0. EDA용 데이터 준비

In [ ]:
eda_2024_df = analysis_2024_core_df.copy()
eda_2024_check_df = analysis_2024_check_df.copy()
eda_long_df = analysis_long_core_df.copy()

# 자치구 정렬 순서
gu_order_2024 = ["중구", "동구", "서구", "남구", "북구", "수성구", "달서구", "달성군", "군위군"]
gu_order_long = ["중구", "동구", "서구", "남구", "북구", "수성구", "달서구", "달성군"]

eda_2024_df["자치구"] = pd.Categorical(eda_2024_df["자치구"], categories=gu_order_2024, ordered=True)
eda_2024_df = eda_2024_df.sort_values("자치구").reset_index(drop=True)

eda_2024_check_df["자치구"] = pd.Categorical(eda_2024_check_df["자치구"], categories=gu_order_2024, ordered=True)
eda_2024_check_df = eda_2024_check_df.sort_values("자치구").reset_index(drop=True)

eda_long_df["자치구"] = pd.Categorical(eda_long_df["자치구"], categories=gu_order_long, ordered=True)
eda_long_df = eda_long_df.sort_values(["연도", "자치구"]).reset_index(drop=True)

print("[eda_2024_df]")
display(eda_2024_df.head())

print("\n[eda_2024_check_df]")
display(eda_2024_check_df.head())

print("\n[eda_long_df]")
display(eda_long_df.head())

### 5-1. 2024 기술통계량 확인

In [ ]:
eda_numeric_cols = [
    "연평균인구",
    "전체범죄건수",
    "5대범죄건수",
    "CCTV총량",
    "자치구총면적_km2",
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "인구10만명당_CCTV수",
    "km2당_CCTV수",
    "범죄1건당_CCTV수",
    "5대범죄1건당_CCTV수",
    "5대범죄비중",
    "폭력비중",
    "방범CCTV비중",
    "주거지역비율",
    "상업지역비율",
    "공업지역비율",
]

eda_numeric_cols = [c for c in eda_numeric_cols if c in eda_2024_df.columns]

eda_desc_df = eda_2024_df[eda_numeric_cols].describe().T
display(eda_desc_df)

### 5-2. 자치구별 기초 비교

##### 5-2-1. 자치구별 기초 비교

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1) 연평균인구
axes[0, 0].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["연평균인구"])
axes[0, 0].set_title("2024 자치구별 연평균인구")
axes[0, 0].tick_params(axis="x", rotation=45)

# 2) 전체범죄건수
axes[0, 1].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["전체범죄건수"])
axes[0, 1].set_title("2024 자치구별 전체범죄건수")
axes[0, 1].tick_params(axis="x", rotation=45)

# 3) 5대범죄건수
axes[1, 0].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["5대범죄건수"])
axes[1, 0].set_title("2024 자치구별 5대범죄건수")
axes[1, 0].tick_params(axis="x", rotation=45)

# 4) CCTV총량
axes[1, 1].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["CCTV총량"])
axes[1, 1].set_title("2024 자치구별 CCTV 총량")
axes[1, 1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

##### 5-2-2. 전체범죄율 / 5대범죄율 / CCTV밀도 비교

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["인구10만명당_전체범죄율"])
axes[0].set_title("인구 10만명당 전체범죄율")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["인구10만명당_5대범죄율"])
axes[1].set_title("인구 10만명당 5대범죄율")
axes[1].tick_params(axis="x", rotation=45)

axes[2].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["km2당_CCTV수"])
axes[2].set_title("km²당 CCTV 수")
axes[2].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

### 5-3. 전체 범죄와 5대범죄 구조 비교

##### 5-3-1. 전체 범죄와 5대범죄 구조 비교

In [ ]:
crime_structure_cols = ["자치구", "전체범죄건수", "5대범죄건수", "5대범죄비중", "폭력비중"]
crime_structure_cols = [c for c in crime_structure_cols if c in eda_2024_df.columns]

crime_structure_df = eda_2024_df[crime_structure_cols].copy()
display(crime_structure_df)

##### 5-3-2. 5대범죄 비중 / 폭력 비중 시각화

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["5대범죄비중"])
axes[0].set_title("전체 범죄 대비 5대범죄 비중")
axes[0].tick_params(axis="x", rotation=45)

axes[1].bar(eda_2024_df["자치구"].astype(str), eda_2024_df["폭력비중"])
axes[1].set_title("5대범죄 내 폭력 비중")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

##### 5-3-3. 5대범죄 세부 구성(살인/강도/성범죄/절도/폭력) 누적막대그래프

In [ ]:
five_cols = ["살인", "강도", "성범죄", "절도", "폭력"]
five_stack_df = eda_2024_df[["자치구"] + five_cols].copy().set_index("자치구")

ax = five_stack_df.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)
ax.set_title("2024 자치구별 5대범죄 구성")
ax.set_xlabel("자치구")
ax.set_ylabel("건수")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5-4. CCTV 설치목적 비교

##### 5-4-1. CCTV 설치목적 비교

In [ ]:
cctv_purpose_cols = [
    "방범",
    "교통관리",
    "어린이 보호구역",
    "재난재해",
    "도시공원 놀이터",
    "쓰레기 투기감시",
    "청사, 시설, 문화재관리",
    "미분류",
]

cctv_purpose_df = eda_2024_df[["자치구"] + cctv_purpose_cols].copy()
display(cctv_purpose_df)

##### 5-4-2. 설치목적별 절대량 누적막대그래프

In [ ]:
ax = cctv_purpose_df.set_index("자치구").plot(
    kind="bar",
    stacked=True,
    figsize=(13, 6)
)
ax.set_title("2024 자치구별 CCTV 설치목적 구성(절대량)")
ax.set_xlabel("자치구")
ax.set_ylabel("카메라수")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

##### 5-4-3. 설치목적 비중 비교(핵심 목적 중심)

In [ ]:
purpose_ratio_cols = [
    "방범CCTV비중",
    "교통관리CCTV비중",
    "재난재해CCTV비중",
    "어린이보호구역CCTV비중",
    "청사시설문화재관리CCTV비중",
    "미분류CCTV비중",
]
purpose_ratio_cols = [c for c in purpose_ratio_cols if c in eda_2024_df.columns]

purpose_ratio_df = eda_2024_df[["자치구"] + purpose_ratio_cols].copy()
display(purpose_ratio_df)

##### 5-4-4. 방범 CCTV 비중 단독 확인

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(eda_2024_df["자치구"].astype(str), eda_2024_df["방범CCTV비중"])
plt.title("2024 자치구별 방범 CCTV 비중")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5-5. 용도지역 특성 비교

##### 5-5-1. 용도지역 특성 비교

In [ ]:
landuse_ratio_df = eda_2024_df[
    ["자치구", "주거지역비율", "상업지역비율", "공업지역비율"]
].copy()

display(landuse_ratio_df)

##### 5-5-2. 용도지역 비율 누적막대그래프

In [ ]:
ax = landuse_ratio_df.set_index("자치구").plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6)
)
ax.set_title("2024 자치구별 주거/상업/공업지역 비율")
ax.set_xlabel("자치구")
ax.set_ylabel("비율")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### 5-6. 변수 간 관계 탐색

##### 5-6-1. 인구 10만명당 5대범죄율 vs 인구 10만명당 CCTV 수

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(eda_2024_df["인구10만명당_CCTV수"], eda_2024_df["인구10만명당_5대범죄율"])
for _, row in eda_2024_df.iterrows():
    plt.text(row["인구10만명당_CCTV수"], row["인구10만명당_5대범죄율"], str(row["자치구"]), fontsize=9)
plt.xlabel("인구 10만명당 CCTV 수")
plt.ylabel("인구 10만명당 5대범죄율")
plt.title("5대범죄율과 CCTV 보급 수준")
plt.tight_layout()
plt.show()

##### 5-6-2. 인구 10만명당 5대범죄율 vs km2당 CCTV 수

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(eda_2024_df["km2당_CCTV수"], eda_2024_df["인구10만명당_5대범죄율"])
for _, row in eda_2024_df.iterrows():
    plt.text(row["km2당_CCTV수"], row["인구10만명당_5대범죄율"], str(row["자치구"]), fontsize=9)
plt.xlabel("km²당 CCTV 수")
plt.ylabel("인구 10만명당 5대범죄율")
plt.title("5대범죄율과 공간적 CCTV 밀도")
plt.tight_layout()
plt.show()

##### 5-6-3. 인구 10만명당 5대범죄율 vs 방범 CCTV 비중

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(eda_2024_df["방범CCTV비중"], eda_2024_df["인구10만명당_5대범죄율"])
for _, row in eda_2024_df.iterrows():
    plt.text(row["방범CCTV비중"], row["인구10만명당_5대범죄율"], str(row["자치구"]), fontsize=9)
plt.xlabel("방범 CCTV 비중")
plt.ylabel("인구 10만명당 5대범죄율")
plt.title("5대범죄율과 방범 CCTV 비중")
plt.tight_layout()
plt.show()

##### 5-6-4. 인구 10만명당 전체범죄율 vs 상업지역비율

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(eda_2024_df["상업지역비율"], eda_2024_df["인구10만명당_전체범죄율"])
for _, row in eda_2024_df.iterrows():
    plt.text(row["상업지역비율"], row["인구10만명당_전체범죄율"], str(row["자치구"]), fontsize=9)
plt.xlabel("상업지역 비율")
plt.ylabel("인구 10만명당 전체범죄율")
plt.title("전체범죄율과 상업지역 비율")
plt.tight_layout()
plt.show()

##### 5-6-5. 인구 10만명당 전체범죄율 vs 주거지역비율

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(eda_2024_df["주거지역비율"], eda_2024_df["인구10만명당_전체범죄율"])
for _, row in eda_2024_df.iterrows():
    plt.text(row["주거지역비율"], row["인구10만명당_전체범죄율"], str(row["자치구"]), fontsize=9)
plt.xlabel("주거지역 비율")
plt.ylabel("인구 10만명당 전체범죄율")
plt.title("전체범죄율과 주거지역 비율")
plt.tight_layout()
plt.show()

### 5-7. 상관행렬 확인

##### 5-7-1. 상관행렬 확인

In [ ]:
corr_cols = [
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "인구10만명당_CCTV수",
    "km2당_CCTV수",
    "범죄1건당_CCTV수",
    "5대범죄1건당_CCTV수",
    "방범CCTV비중",
    "교통관리CCTV비중",
    "재난재해CCTV비중",
    "주거지역비율",
    "상업지역비율",
    "공업지역비율",
]
corr_cols = [c for c in corr_cols if c in eda_2024_df.columns]

corr_df = eda_2024_df[corr_cols].corr()
display(corr_df)

##### 5-7-2. matplotlib 기반 상관행렬 히트맵

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr_df.values)

ax.set_xticks(range(len(corr_df.columns)))
ax.set_yticks(range(len(corr_df.index)))
ax.set_xticklabels(corr_df.columns, rotation=90)
ax.set_yticklabels(corr_df.index)

for i in range(len(corr_df.index)):
    for j in range(len(corr_df.columns)):
        ax.text(j, i, f"{corr_df.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("2024 핵심 변수 상관행렬")
fig.colorbar(im)
plt.tight_layout()
plt.show()

### 5-8. 장기 비교(군위군 제외) 추이 확인

##### 5-8-1. 연도별 대구 8개 자치구 평균 전체 범죄율

In [ ]:
yearly_avg_total = (
    eda_long_df.groupby("연도", as_index=False)["인구10만명당_전체범죄율"]
    .mean()
)

plt.figure(figsize=(8, 4))
plt.plot(yearly_avg_total["연도"], yearly_avg_total["인구10만명당_전체범죄율"], marker="o")
plt.title("연도별 평균 전체범죄율(군위군 제외)")
plt.xlabel("연도")
plt.ylabel("인구 10만명당 전체범죄율")
plt.xticks(yearly_avg_total["연도"])
plt.tight_layout()
plt.show()

##### 5-8-2. 연도별 평균 5대범죄율

In [ ]:
yearly_avg_five = (
    eda_long_df.groupby("연도", as_index=False)["인구10만명당_5대범죄율"]
    .mean()
)

plt.figure(figsize=(8, 4))
plt.plot(yearly_avg_five["연도"], yearly_avg_five["인구10만명당_5대범죄율"], marker="o")
plt.title("연도별 평균 5대범죄율(군위군 제외)")
plt.xlabel("연도")
plt.ylabel("인구 10만명당 5대범죄율")
plt.xticks(yearly_avg_five["연도"])
plt.tight_layout()
plt.show()

##### 5-8-3. 자치구별 전체범죄율 장기 추이

In [ ]:
pivot_total_rate = eda_long_df.pivot(index="연도", columns="자치구", values="인구10만명당_전체범죄율")

ax = pivot_total_rate.plot(figsize=(10, 6), marker="o")
ax.set_title("자치구별 전체범죄율 장기 추이(군위군 제외)")
ax.set_xlabel("연도")
ax.set_ylabel("인구 10만명당 전체범죄율")
plt.xticks(pivot_total_rate.index)
plt.tight_layout()
plt.show()

### 5-9. 2024 지도 시각화

In [ ]:
eda_map_gdf = geo_2024_df.merge(
    eda_2024_df.drop(columns=["자치구총면적_km2"]),
    on="자치구",
    how="left"
)

print("[eda_map_gdf]")
display(eda_map_gdf[["자치구", "인구10만명당_전체범죄율", "인구10만명당_5대범죄율", "km2당_CCTV수"]].head())

##### 5-9-1. 전체범죄율 지도

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
eda_map_gdf.plot(column="인구10만명당_전체범죄율", legend=True, ax=ax, edgecolor="black")
for _, row in eda_map_gdf.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    ax.text(x, y, row["자치구"], fontsize=9, ha="center")
ax.set_title("2024 자치구별 인구 10만명당 전체범죄율")
ax.set_axis_off()
plt.tight_layout()
plt.show()

##### 5-9-2. 5대범죄율 지도

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
eda_map_gdf.plot(column="인구10만명당_5대범죄율", legend=True, ax=ax, edgecolor="black")
for _, row in eda_map_gdf.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    ax.text(x, y, row["자치구"], fontsize=9, ha="center")
ax.set_title("2024 자치구별 인구 10만명당 5대범죄율")
ax.set_axis_off()
plt.tight_layout()
plt.show()

##### 5-9-3. km2당 CCTV 수 지도

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 8))
eda_map_gdf.plot(column="km2당_CCTV수", legend=True, ax=ax, edgecolor="black")
for _, row in eda_map_gdf.iterrows():
    x, y = row.geometry.centroid.x, row.geometry.centroid.y
    ax.text(x, y, row["자치구"], fontsize=9, ha="center")
ax.set_title("2024 자치구별 km²당 CCTV 수")
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 5-10. EDA 결과 저장용 정리 테이블

##### 5-10-1. EDA 결과 저장용 정리 테이블

In [ ]:
eda_summary_cols = [
    "자치구",
    "연평균인구",
    "전체범죄건수",
    "5대범죄건수",
    "CCTV총량",
    "자치구총면적_km2",
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "인구10만명당_CCTV수",
    "km2당_CCTV수",
    "5대범죄비중",
    "폭력비중",
    "방범CCTV비중",
    "상업지역비율",
    "주거지역비율",
    "공업지역비율",
]
eda_summary_cols = [c for c in eda_summary_cols if c in eda_2024_df.columns]

eda_summary_2024_df = eda_2024_df[eda_summary_cols].copy()
display(eda_summary_2024_df.sort_values("자치구").reset_index(drop=True))

##### 5-10-2. 저장

In [ ]:
eda_summary_2024_df.to_csv(
    os.path.join(output_path, "eda_summary_2024.csv"),
    index=False,
    encoding="utf-8-sig"
)

corr_df.to_csv(
    os.path.join(output_path, "eda_corr_2024.csv"),
    encoding="utf-8-sig"
)

print("EDA 요약 결과 저장 완료")

# 6. 결과 정리 및 인사이트 도출

### 6-0. 결과 정리용 데이터 준비

In [ ]:
result_2024_df = (
    analysis_2024_core_df
    .merge(analysis_2024_check_df, on="자치구", how="left")
    .copy()
)
if "좌표유효CCTV수_x" in result_2024_df.columns and "좌표유효CCTV수_y" in result_2024_df.columns:
    result_2024_df["좌표유효CCTV수"] = result_2024_df["좌표유효CCTV수_y"]
    result_2024_df = result_2024_df.drop(columns=["좌표유효CCTV수_x", "좌표유효CCTV수_y"])


result_long_df = analysis_long_core_df.copy()

# 좌표/품질 보조 지표
result_2024_df["원본좌표CCTV비중"] = safe_divide(
    result_2024_df["원본좌표채택CCTV수"],
    result_2024_df["CCTV총량"]
)

result_2024_df["카카오좌표CCTV비중"] = safe_divide(
    result_2024_df["카카오좌표채택CCTV수"],
    result_2024_df["CCTV총량"]
)

# CCTV 총량 해석 주의 등급
result_2024_df["CCTV해석주의등급"] = np.select(
    [
        result_2024_df["카메라수결측대체비율"] >= 0.8,
        result_2024_df["카메라수결측대체비율"] >= 0.3,
    ],
    [
        "높음",
        "중간",
    ],
    default="낮음"
)

# 좌표 활용 가능성 등급
result_2024_df["좌표활용등급"] = np.select(
    [
        result_2024_df["좌표유효CCTV비율"] >= 0.90,
        result_2024_df["좌표유효CCTV비율"] >= 0.70,
    ],
    [
        "양호",
        "보통",
    ],
    default="제한적"
)

print("[result_2024_df]")
display(result_2024_df.sort_values("자치구").reset_index(drop=True))

print("\n[result_long_df]")
display(result_long_df.head(20))

### 6-1. 2024 핵심 지표 순위 정리

In [ ]:
ranking_df = result_2024_df[
    [
        "자치구",
        "연평균인구",
        "전체범죄건수",
        "5대범죄건수",
        "CCTV총량",
        "인구10만명당_전체범죄율",
        "인구10만명당_5대범죄율",
        "인구10만명당_CCTV수",
        "km2당_CCTV수",
        "방범CCTV비중",
        "주거지역비율",
        "상업지역비율",
        "공업지역비율",
        "좌표유효CCTV비율",
        "카메라수결측대체비율",
        "CCTV해석주의등급",
        "좌표활용등급",
    ]
].copy()

rank_desc_cols = [
    "전체범죄건수",
    "5대범죄건수",
    "CCTV총량",
    "인구10만명당_전체범죄율",
    "인구10만명당_5대범죄율",
    "인구10만명당_CCTV수",
    "km2당_CCTV수",
    "방범CCTV비중",
    "상업지역비율",
]

for col in rank_desc_cols:
    ranking_df[f"{col}_순위"] = ranking_df[col].rank(method="min", ascending=False)

print("[ranking_df]")
display(ranking_df.sort_values("인구10만명당_전체범죄율_순위").reset_index(drop=True))

### 6-2. 자치구 우선관리 점수 산정

##### 기준.
##### - 높을수록 위험:
- 인구 10만명당 전체범죄율
- 인구 10만명당 5대범죄율
- 상업지역 비율

##### - 낮을수록 관리 필요:
- km2당 CCTV 수
- 방범 CCTV 비중

In [ ]:
priority_df = result_2024_df[
    [
        "자치구",
        "인구10만명당_전체범죄율",
        "인구10만명당_5대범죄율",
        "km2당_CCTV수",
        "방범CCTV비중",
        "상업지역비율",
        "좌표유효CCTV비율",
        "카메라수결측대체비율",
        "CCTV해석주의등급",
        "좌표활용등급",
    ]
].copy()

priority_df["전체범죄율_위험순위"] = priority_df["인구10만명당_전체범죄율"].rank(method="min", ascending=False)
priority_df["5대범죄율_위험순위"] = priority_df["인구10만명당_5대범죄율"].rank(method="min", ascending=False)
priority_df["상업지역비율_위험순위"] = priority_df["상업지역비율"].rank(method="min", ascending=False)

priority_df["CCTV밀도_부족순위"] = priority_df["km2당_CCTV수"].rank(method="min", ascending=True)
priority_df["방범비중_부족순위"] = priority_df["방범CCTV비중"].rank(method="min", ascending=True)

priority_df["우선관리점수"] = (
    priority_df["전체범죄율_위험순위"] +
    priority_df["5대범죄율_위험순위"] +
    priority_df["상업지역비율_위험순위"] +
    priority_df["CCTV밀도_부족순위"] +
    priority_df["방범비중_부족순위"]
)

priority_df["관리등급"] = np.select(
    [
        priority_df["우선관리점수"] >= 28,
        priority_df["우선관리점수"] >= 23,
    ],
    [
        "우선관리",
        "주의관리",
    ],
    default="상대적 안정"
)

priority_df["최종해석메모"] = np.select(
    [
        (priority_df["CCTV해석주의등급"] == "높음") & (priority_df["좌표활용등급"] == "제한적"),
        (priority_df["CCTV해석주의등급"] == "높음"),
        (priority_df["좌표활용등급"] == "제한적"),
    ],
    [
        "CCTV 총량·좌표 모두 해석주의",
        "CCTV 총량 해석주의",
        "좌표 기반 해석 제한",
    ],
    default="핵심지표 해석 가능"
)

print("[priority_df]")
display(priority_df.sort_values(["우선관리점수", "인구10만명당_전체범죄율"], ascending=[False, False]).reset_index(drop=True))

### 6-2A. 좌표 기반 CCTV 활용 가능성 점검

In [ ]:
coord_use_df = result_2024_df[
    [
        "자치구",
        "CCTV총량",
        "원본좌표채택CCTV수",
        "카카오좌표채택CCTV수",
        "좌표유효CCTV비율",
        "카메라수결측대체비율",
        "CCTV해석주의등급",
        "좌표활용등급",
    ]
].copy()

coord_use_df["원본좌표CCTV비중"] = safe_divide(
    coord_use_df["원본좌표채택CCTV수"],
    coord_use_df["CCTV총량"]
)

coord_use_df["카카오보완CCTV비중"] = safe_divide(
    coord_use_df["카카오좌표채택CCTV수"],
    coord_use_df["CCTV총량"]
)

coord_use_df["좌표기반활용판단"] = np.select(
    [
        (coord_use_df["좌표유효CCTV비율"] >= 0.90) & (coord_use_df["카메라수결측대체비율"] < 0.30),
        (coord_use_df["좌표유효CCTV비율"] >= 0.70) & (coord_use_df["카메라수결측대체비율"] < 0.80),
    ],
    [
        "좌표 활용 양호",
        "좌표 활용 가능",
    ],
    default="좌표 활용 제한적"
)

print("[coord_use_df]")
display(coord_use_df.sort_values(["좌표기반활용판단", "좌표유효CCTV비율"], ascending=[True, False]).reset_index(drop=True))

### 6-3. 자치구 유형화

In [ ]:
type_df = result_2024_df[
    [
        "자치구",
        "인구10만명당_5대범죄율",
        "km2당_CCTV수",
        "방범CCTV비중",
        "상업지역비율",
    ]
].copy()

# 중앙값 기준 분류
five_median = type_df["인구10만명당_5대범죄율"].median()
cctv_median = type_df["km2당_CCTV수"].median()
defense_median = type_df["방범CCTV비중"].median()
commercial_median = type_df["상업지역비율"].median()

type_df["5대범죄수준"] = np.where(type_df["인구10만명당_5대범죄율"] >= five_median, "고", "저")
type_df["CCTV밀도수준"] = np.where(type_df["km2당_CCTV수"] >= cctv_median, "고", "저")
type_df["방범비중수준"] = np.where(type_df["방범CCTV비중"] >= defense_median, "고", "저")
type_df["상업지역수준"] = np.where(type_df["상업지역비율"] >= commercial_median, "고", "저")

def classify_area(row):
    if row["5대범죄수준"] == "고" and row["상업지역수준"] == "고":
        return "상업지역 중심 위험형"
    elif row["5대범죄수준"] == "고" and row["CCTV밀도수준"] == "저":
        return "고범죄·저CCTV형"
    elif row["5대범죄수준"] == "고" and row["방범비중수준"] == "저":
        return "고범죄·저방범형"
    elif row["5대범죄수준"] == "저" and row["CCTV밀도수준"] == "고":
        return "저범죄·고CCTV형"
    else:
        return "혼합형"

type_df["자치구유형"] = type_df.apply(classify_area, axis=1)

display(type_df.sort_values("자치구").reset_index(drop=True))

### 6-4. 장기 변화량 정의(2019 -> 2024)

In [ ]:
long_change_base = result_long_df[
    result_long_df["연도"].isin([2019, 2024])
][
    [
        "연도",
        "자치구",
        "인구10만명당_전체범죄율",
        "인구10만명당_5대범죄율",
        "주거지역비율",
        "상업지역비율",
        "공업지역비율",
    ]
].copy()

long_change_pivot = long_change_base.pivot(index="자치구", columns="연도")
long_change_pivot.columns = [f"{col}_{year}" for col, year in long_change_pivot.columns]
long_change_pivot = long_change_pivot.reset_index()

# 변화량 계산
long_change_pivot["전체범죄율_변화량"] = (
    long_change_pivot["인구10만명당_전체범죄율_2024"]
    - long_change_pivot["인구10만명당_전체범죄율_2019"]
)

long_change_pivot["5대범죄율_변화량"] = (
    long_change_pivot["인구10만명당_5대범죄율_2024"]
    - long_change_pivot["인구10만명당_5대범죄율_2019"]
)

display(long_change_pivot.sort_values("전체범죄율_변화량"))

### 6-5. 핵심 결과 요약 테이블 만들기

In [ ]:
summary_2024_df = result_2024_df[
    [
        "자치구",
        "인구10만명당_전체범죄율",
        "인구10만명당_5대범죄율",
        "인구10만명당_CCTV수",
        "km2당_CCTV수",
        "방범CCTV비중",
        "상업지역비율",
        "주거지역비율",
        "공업지역비율",
    ]
].copy()

summary_2024_df = summary_2024_df.merge(
    priority_df[["자치구", "우선관리점수", "관리등급"]],
    on="자치구",
    how="left"
)

summary_2024_df = summary_2024_df.merge(
    type_df[["자치구", "자치구유형"]],
    on="자치구",
    how="left"
)

display(summary_2024_df.sort_values("우선관리점수", ascending=False).reset_index(drop=True))

### 6-6. 결과 저장

In [ ]:
result_2024_df.to_csv(
    os.path.join(output_path, "result_2024_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

result_long_df.to_csv(
    os.path.join(output_path, "result_long_summary.csv"),
    index=False,
    encoding="utf-8-sig"
)

ranking_df.to_csv(
    os.path.join(output_path, "result_2024_ranking.csv"),
    index=False,
    encoding="utf-8-sig"
)

priority_df.to_csv(
    os.path.join(output_path, "result_2024_priority.csv"),
    index=False,
    encoding="utf-8-sig"
)

coord_use_df.to_csv(
    os.path.join(output_path, "result_2024_coordinate_quality.csv"),
    index=False,
    encoding="utf-8-sig"
)

print("결과 정리 및 인사이트 도출용 테이블 저장 완료")